# PRESAGE — reproducible pipeline (Google Colab, free tier)

Gap-aware selective state-space learning with competing-risk onset forecasting and conformal warning guarantees for pre-symptomatic crop stress.

This notebook rebuilds both data tiers from public sources, trains PRESAGE and all baselines, computes every metric and regenerates the figures of the manuscript.

* **Runtime:** CPU is sufficient. `MODE = "quick"` (one seed, main models) takes roughly 1.5–2 h on the free tier; `MODE = "full"` reproduces the paper (all seeds, ablations, spatial and real-field folds) and needs about 8–9 h, so run it across sessions — finished runs are skipped automatically.
* **Data:** OSMER ARPA FVG daily weather (CC BY-SA 3.0 IT) and the FBK OpenIoT soil-moisture dataset (CC BY-NC-ND 4.0), both downloaded from the Hugging Face Hub; WOFOST 8.1 crop parameters from GitHub.
* Results are written to `/content/presage/results`, figures to `/content/presage/figures`.

In [ ]:
MODE = "quick"   # "quick" or "full"
import os
os.makedirs('/content/presage', exist_ok=True)
os.makedirs('/content/data', exist_ok=True)
%cd /content/presage

## 1. Environment

In [ ]:
!pip -q install pcse==6.1.0 lifelines pyarrow huggingface_hub
!git clone -q --depth 1 -b wofost81 https://github.com/ajwdewit/WOFOST_crop_parameters.git /content/data/wofost81 || true
!apt-get -qq install -y fonts-liberation > /dev/null
import torch, pcse; print(torch.__version__, pcse.__version__)

## 2. Data download

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download('jotone/arpa-osmer-weatherstations', repo_type='dataset', local_dir='/content/data/fvg_raw',
                  allow_patterns=['daily/*', 'stations.parquet'])
snapshot_download('it4lia/soil_moisture_dataset', repo_type='dataset', local_dir='/content/data/it4lia',
                  allow_patterns=['*.parquet'], ignore_patterns=['forecasted_*'])
import glob, shutil
os.makedirs('/content/data/fvg/daily', exist_ok=True)
for f in glob.glob('/content/data/fvg_raw/daily/*/aggregates.parquet'):
    shutil.copy(f, f"/content/data/fvg/daily/{f.split('/')[-2]}.parquet")
shutil.copy('/content/data/fvg_raw/stations.parquet', '/content/data/fvg/stations.parquet')
print(len(glob.glob('/content/data/fvg/daily/*.parquet')), 'station files')

## 3. Source modules
Each cell writes one module of the project.

In [ ]:
%%writefile sim_wofost.py
"""Real-weather-driven WOFOST 8.1 (water + nitrogen limited) simulation for one field-season."""
import datetime as dt
import math
import numpy as np
import pandas as pd
from pcse.base import WeatherDataProvider, WeatherDataContainer, ParameterProvider
from pcse.input import YAMLCropDataProvider
from pcse.util import reference_ET
from pcse.models import Wofost81_NWLP_CWB_CNB, Wofost81_PP
import yaml

CROPDIR = "/content/data/wofost81"

SOILS = {
    # name: SMW, SMFCF, SM0, RDMSOL (cm)
    "sandy_loam": dict(SMW=0.09, SMFCF=0.23, SM0=0.40, RDMSOL=80.),
    "silt_loam": dict(SMW=0.14, SMFCF=0.34, SM0=0.46, RDMSOL=110.),
    "clay_loam": dict(SMW=0.21, SMFCF=0.41, SM0=0.50, RDMSOL=100.),
}


def sat_vp(t):
    return 0.6108 * math.exp(17.27 * t / (t + 237.3))  # kPa


class FrameWeather(WeatherDataProvider):
    def __init__(self, df, lat, lon, elev):
        WeatherDataProvider.__init__(self)
        self.description = "station"
        for r in df.itertuples():
            day = r.date.date()
            tmin, tmax = float(r.tmin), float(r.tmax)
            if tmin > tmax:
                tmin, tmax = tmax, tmin
            tavg = 0.5 * (tmin + tmax)
            vap = max(0.2, r.rh / 100.0 * sat_vp(tavg)) * 10.0  # hPa
            irrad = float(r.rg) * 1000.0  # kJ m-2 d-1 -> J m-2 d-1
            wind = max(0.3, float(r.wind) / 3.6)  # km/h -> m/s at 10 m
            wind2 = wind * 4.87 / math.log(67.8 * 10 - 5.42)
            e0, es0, et0 = reference_ET(day, lat, elev, tmin, tmax, irrad, vap, wind2, 0.29, 0.49, ETMODEL="PM")
            wdc = WeatherDataContainer(LAT=lat, LON=lon, ELEV=elev, DAY=day, IRRAD=irrad, TMIN=tmin, TMAX=tmax,
                                       VAP=vap, RAIN=min(24.9, max(0.0, float(r.rain) / 10.0)), E0=e0 / 10., ES0=es0 / 10., ET0=et0 / 10.,
                                       WIND=wind2, TEMP=tavg, SNOWDEPTH=0.0)
            self._store_WeatherDataContainer(wdc, day)


_CROPD = None


def crop_provider():
    global _CROPD
    if _CROPD is None:
        _CROPD = YAMLCropDataProvider(fpath=CROPDIR)
    return _CROPD


def agro(crop, variety, sow, n_events, irr_events, end):
    timed = []
    if n_events:
        timed.append({"event_signal": "apply_n", "name": "N", "comment": "",
                      "events_table": [{d: {"N_amount": a, "N_recovery": 0.7}} for d, a in n_events]})
    if irr_events:
        timed.append({"event_signal": "irrigate", "name": "irr", "comment": "",
                      "events_table": [{d: {"amount": a, "efficiency": 0.75}} for d, a in irr_events]})
    a = [{sow - dt.timedelta(days=1): {"CropCalendar": {"crop_name": crop, "variety_name": variety,
                                                         "crop_start_date": sow, "crop_start_type": "sowing",
                                                         "crop_end_date": end, "crop_end_type": "earliest",
                                                         "max_duration": 200},
                                        "TimedEvents": timed or None, "StateEvents": None}}]
    return a


def run_season(wdp, crop, variety, sow, soil, n_events, irr_events, nsoil, wav, potential=False):
    cp = crop_provider()
    cp.set_active_crop(crop, variety)
    s = dict(SOILS[soil])
    soildata = dict(SMW=s["SMW"], SMFCF=s["SMFCF"], SM0=s["SM0"], CRAIRC=0.06, SOPE=10., KSUB=10.,
                    RDMSOL=s["RDMSOL"], K0=10.,
                    NSOILBASE=nsoil, NSOILBASE_FR=0.025, NAVAILI=0.3 * nsoil, BG_N_SUPPLY=0.091)
    site = dict(WAV=wav, IFUNRN=0, NOTINF=0, SSI=0, SSMAX=0, SMLIM=s["SM0"], CO2=410.)
    params = ParameterProvider(cropdata=cp, soildata=soildata, sitedata=site)
    end = sow + dt.timedelta(days=199)
    am = agro(crop, variety, sow, n_events, irr_events, end)
    model = (Wofost81_PP if potential else Wofost81_NWLP_CWB_CNB)(params, wdp, am)
    model.run_till_terminate()
    out = pd.DataFrame(model.get_output())
    return out, cp


In [ ]:
%%writefile disease.py
"""Weather-driven foliar epidemic with latent and infectious compartments (daily step).

Structure follows the site-based compartment logic of EPIRICE-type models: healthy (H), latent (L),
infectious (I) and removed (R) leaf-area fractions, with infection efficiency modulated by a
temperature response and a leaf-wetness proxy, and by host age and cultivar resistance.
"""
import numpy as np

PATHO = {
    # crop: Tmin, Topt, Tmax, latent period (d), infectious period (d), basic daily rate
    "maize": dict(tmin=10., topt=24., tmax=32., p=9, i=14, rc=1.3),
    "soybean": dict(tmin=12., topt=25., tmax=33., p=8, i=12, rc=1.3),
}


def temp_response(t, tmin, topt, tmax):
    t = np.clip(t, tmin, tmax)
    a = (t - tmin) / (topt - tmin)
    b = (tmax - t) / (tmax - topt)
    return np.clip(a ** 1.0 * b ** ((tmax - topt) / (topt - tmin)), 0, 1)


def wetness(rain, rh):
    """Daily leaf-wetness proxy in [0, 1] from precipitation and mean relative humidity."""
    w = np.clip((rh - 70.) / 20., 0., 1.) * 0.7
    w = np.where(rain >= 1.0, 1.0, w)
    return w


def epidemic(tavg, rain, rh, dvs, arrival, resist, crop, rng):
    """Returns daily latent fraction L, visible severity S (I+R) and the infection-pressure index."""
    p = PATHO[crop]
    n = len(tavg)
    fT = temp_response(tavg, p["tmin"], p["topt"], p["tmax"])
    fW = wetness(rain, rh)
    # host age: susceptibility rises after canopy closure, declines at senescence
    fA = np.clip(np.where(dvs < 0.3, dvs / 0.3 * 0.5, np.where(dvs < 1.4, 1.0, 1.0 - (dvs - 1.4))), 0.1, 1.0)
    pressure = fT * fW
    lat = np.zeros(n + p["p"] + p["i"] + 2)
    inf = np.zeros_like(lat)
    L = np.zeros(n); S = np.zeros(n)
    H = 1.0; Itot = 0.0; R = 0.0
    queue_lat = []  # (day_becomes_infectious, amount)
    queue_inf = []  # (day_removed, amount)
    Lcur = 0.0
    for d in range(n):
        if arrival is not None and d == arrival:
            amt = 1e-3
            queue_lat.append((d + p["p"], amt)); Lcur += amt; H -= amt
        # transitions
        for q in [x for x in queue_lat if x[0] == d]:
            queue_lat.remove(q); Lcur -= q[1]; Itot += q[1]; queue_inf.append((d + p["i"], q[1]))
        for q in [x for x in queue_inf if x[0] == d]:
            queue_inf.remove(q); Itot -= q[1]; R += q[1]
        # new infections (logistic limitation by healthy fraction)
        rate = p["rc"] * pressure[d] * fA[d] * (1. - resist)
        new = min(H, rate * Itot * H)
        if new > 0:
            pdays = p["p"] + int(rng.integers(-1, 2))
            queue_lat.append((d + pdays, new)); Lcur += new; H -= new
        L[d] = Lcur; S[d] = Itot + R
    return L, S, pressure


In [ ]:
%%writefile build_sim.py
"""Builds the real-weather-driven multimodal crop stress onset benchmark (FVG stations).

Each field-season = (station, year, management scenario). WOFOST 8.1 (water + N limited) provides crop
physiology; a compartment epidemic provides latent / visible infection; sensor models then generate
asynchronous observation tokens. Output: data/sim_fieldseasons.pkl
"""
import datetime as dt
import glob
import math
import os
import pickle
import sys
import warnings
import zlib
from multiprocessing import Pool

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.dirname(__file__))
from sim_wofost import FrameWeather, run_season, crop_provider, SOILS  # noqa
from disease import epidemic  # noqa

FVG = "/content/data/fvg"
OUT = "/content/presage/data"
REQ = ["rr", "t180_min", "t180_max", "rh_med", "v10_med", "rg"]
N_SCEN = int(os.environ.get("N_SCEN", "6"))
KAPPA_BIO = float(os.environ.get("KAPPA_BIO", "1.0"))  # strength of pre-symptomatic signatures of latent infection

MOD = {"static": 0, "weather": 1, "soil": 2, "optical": 3, "thermal": 4, "mgmt": 5, "trap": 6}
TOD = {"static": 0.0, "mgmt": 0.30, "trap": 0.40, "optical": 0.45, "soil": 0.50, "thermal": 0.55, "weather": 0.99}


def zone_of(alt, lat):
    if alt < 15:
        return 0  # coastal lowland
    if alt < 120:
        return 1  # alluvial plain
    if alt < 400:
        return 2  # foothill / morainic hills
    return 3  # alpine valley


def station_years():
    st = pd.read_parquet(f"{FVG}/stations.parquet").set_index("code")
    items = []
    for f in sorted(glob.glob(f"{FVG}/daily/*.parquet")):
        d = pd.read_parquet(f)
        code = d.code.iloc[0]
        if code not in st.index:
            continue
        d["date"] = pd.to_datetime(d["date"])
        for yr, g in d.groupby(d.date.dt.year):
            gs = g[(g.date.dt.month >= 4) & (g.date.dt.month <= 9)]
            if len(gs) < 170:
                continue
            if gs[REQ].notna().all(axis=1).mean() < 0.95:
                continue
            w = g[(g.date >= f"{yr}-03-01") & (g.date <= f"{yr}-11-30")].copy()
            if len(w) < 260:
                continue
            items.append((code, int(yr), float(st.loc[code, "alt"]), float(st.loc[code, "lat"]),
                          float(st.loc[code, "lon"]), w))
    return items


def vpd_kpa(tmin, tmax, rh):
    es = 0.5 * (0.6108 * np.exp(17.27 * tmax / (tmax + 237.3)) + 0.6108 * np.exp(17.27 * tmin / (tmin + 237.3)))
    return np.clip(es * (1 - rh / 100.), 0.05, 5.0)


def sustained(mask, k):
    """first index i where mask[i:i+k] all True"""
    run = 0
    for i, m in enumerate(mask):
        run = run + 1 if m else 0
        if run >= k:
            return i - k + 1
    return None


def simulate(args):
    code, yr, alt, lat, lon, w, scen = args
    rng = np.random.default_rng(zlib.crc32(f"{code}|{yr}|{scen}|{KAPPA_BIO}".encode()))
    w = w.rename(columns={"t180_min": "tmin", "t180_max": "tmax", "rh_med": "rh", "v10_med": "wind", "rr": "rain"})
    num = ["tmin", "tmax", "rh", "wind", "rain", "rg"]
    w.loc[(w.rg > 34000) | (w.rg < 0), "rg"] = np.nan  # quality control of implausible records
    w.loc[(w.tmax > 42) | (w.tmax < -25), "tmax"] = np.nan
    w.loc[(w.tmin > 32) | (w.tmin < -30), "tmin"] = np.nan
    w.loc[(w.rh > 100) | (w.rh < 5), "rh"] = np.nan
    w.loc[(w.rain < 0) | (w.rain > 240), "rain"] = np.nan
    w.loc[(w.wind < 0) | (w.wind > 120), "wind"] = np.nan
    miss = w[num].isna().any(axis=1).values
    w[num] = w[num].astype(float).interpolate(limit_direction="both")
    w = w.reset_index(drop=True)
    try:
        wdp = FrameWeather(w, lat, lon, alt)
    except Exception:
        return None
    crop = "maize" if rng.random() < 0.6 else "soybean"
    variety = "Grain_maize_201" if crop == "maize" else "Soybean_901"
    soil = list(SOILS)[rng.integers(3)]
    if crop == "maize":
        sow = dt.date(yr, 4, 5) + dt.timedelta(days=int(rng.integers(0, 35)))
    else:
        sow = dt.date(yr, 4, 25) + dt.timedelta(days=int(rng.integers(0, 35)))
    nsoil = float(rng.uniform(15, 90))
    n_events = []
    if crop == "maize":
        plan = rng.choice(["none", "low", "split"], p=[0.25, 0.35, 0.40])
        if plan == "low":
            n_events = [(sow + dt.timedelta(days=int(rng.integers(20, 40))), float(rng.uniform(30, 70)))]
        elif plan == "split":
            n_events = [(sow + dt.timedelta(days=int(rng.integers(15, 30))), float(rng.uniform(40, 80))),
                        (sow + dt.timedelta(days=int(rng.integers(45, 60))), float(rng.uniform(40, 90)))]
    else:
        if rng.random() < 0.3:
            n_events = [(sow + dt.timedelta(days=int(rng.integers(15, 30))), float(rng.uniform(15, 35)))]
    irr = []
    irr_plan = rng.choice(["rainfed", "calendar", "late"], p=[0.45, 0.35, 0.20])
    if irr_plan != "rainfed":
        start = sow + dt.timedelta(days=int(rng.integers(35, 55) if irr_plan == "calendar" else rng.integers(60, 80)))
        step = int(rng.integers(7, 15))
        amt = float(rng.uniform(1.5, 3.5))  # cm
        d = start
        while d < sow + dt.timedelta(days=130):
            irr.append((d, amt))
            d += dt.timedelta(days=step)
    wav = float(rng.uniform(4, 16))
    try:
        out, cp = run_season(wdp, crop, variety, sow, soil, n_events, irr, nsoil, wav)
    except Exception as e:  # pragma: no cover
        return None
    out["day"] = pd.to_datetime(out["day"])
    out = out[out.DVS.notna()].reset_index(drop=True)
    if len(out) < 60:
        return None
    tb = cp["NMAXLV_TB"]
    xs, ys = tb[0::2], tb[1::2]
    nmaxlv = np.interp(out.DVS.values, xs, ys)
    with np.errstate(divide="ignore", invalid="ignore"):
        ns = np.where(out.WLV.values > 1, out.NamountLV.values / (out.WLV.values * nmaxlv), 1.0)
    ns = np.nan_to_num(ns, nan=1.0)
    # align weather to crop days
    wd = w.set_index(pd.to_datetime(w.date))
    days = out.day
    tavg = 0.5 * (wd.loc[days, "tmin"].values + wd.loc[days, "tmax"].values)
    rain = wd.loc[days, "rain"].values
    rh = wd.loc[days, "rh"].values
    dvs = out.DVS.values
    n = len(out)
    arrival = int(rng.integers(15, min(85, n - 10))) if rng.random() < 0.75 else None
    resist = float(rng.uniform(0.0, 0.6))
    L, S, pressure = epidemic(tavg, rain, rh, dvs, arrival, resist, crop, rng)
    rftra = np.nan_to_num(out.RFTRA.values, nan=1.0)
    lai = np.nan_to_num(out.LAI.values, nan=0.0)
    sm = out.SM.values
    # ---------- onsets ----------
    valid = dvs < 1.6
    end = int(np.argmax(~valid)) if (~valid).any() else n
    start = 20
    def first(mask, k, lo=start):
        m = mask.copy(); m[:lo] = False; m[end:] = False
        return sustained(m, k)
    Tw = first((rftra < 0.8) & (lai > 0.6), 2)
    Tn = first((ns < 0.75) & (dvs > 0.25), 3)
    Tp = first((L + S) >= 4e-3, 1)
    Vw = first((rftra < 0.45) & (lai > 0.6), 2, lo=Tw if Tw is not None else start)
    Vn = first((ns < 0.55) & (dvs > 0.25), 2, lo=Tn if Tn is not None else start)
    Vp = first(S >= 0.01, 1, lo=Tp if Tp is not None else start)
    onsets = {0: Tw, 1: Tn, 2: Tp}
    vis = {0: Vw, 1: Vn, 2: Vp}
    cand = [(t, j) for j, t in onsets.items() if t is not None]
    if cand:
        T, J = min(cand)
    else:
        T, J = None, -1
    # ---------- sensor tokens ----------
    toks = []  # (t, modality, values list)
    tmin_ = wd.loc[days, "tmin"].values; tmax_ = wd.loc[days, "tmax"].values
    rg = wd.loc[days, "rg"].values; wind = wd.loc[days, "wind"].values
    vpd = vpd_kpa(tmin_, tmax_, rh)
    gdd = np.cumsum(np.clip(tavg - (10 if crop == "maize" else 8), 0, None))
    dmiss = pd.Series(miss, index=pd.to_datetime(w.date)).loc[days].values
    soilp = SOILS[soil]
    toks.append((0.0, MOD["static"], [1.0 if crop == "maize" else 0.0, soilp["SMW"], soilp["SMFCF"], soilp["SM0"],
                                       alt / 1000., lat - 46.0]))
    for d in range(n):
        if not dmiss[d]:
            toks.append((d + TOD["weather"], MOD["weather"], [tmin_[d], tmax_[d], rh[d], math.log1p(rain[d]), rg[d] / 1000.,
                                                              wind[d], vpd[d], gdd[d] / 1000.]))
    # soil moisture probe with bias and failure episodes
    bias = rng.normal(0, 0.02)
    dead = 0
    for d in range(n):
        if dead > 0:
            dead -= 1; continue
        if rng.random() < 0.012:
            dead = int(rng.geometric(1 / 9.)); continue
        toks.append((d + TOD["soil"], MOD["soil"], [sm[d] + bias + rng.normal(0, 0.008)]))
    # optical revisit (5-day) with clouds
    phase = int(rng.integers(0, 5))
    Lrel = np.clip(L / 0.01, 0, 1)
    for d in range(phase, n, 5):
        cloudy = rain[d] > 0.5 or (rh[d] > 85 and rng.random() < 0.6) or rng.random() < 0.12
        if cloudy:
            continue
        roll = 1.0 - 0.5 * max(0.0, 0.5 - rftra[d])
        laig = lai[d] * (1 - S[d]) * roll
        fc = 1 - math.exp(-0.55 * laig)
        chl = (np.clip(ns[d], 0.3, 1.1) - 0.3) / 0.8
        ndvi = 0.12 + 0.78 * fc + rng.normal(0, 0.02)
        ndre = 0.04 + (1 - math.exp(-0.5 * laig)) * (0.12 + 0.38 * chl) - KAPPA_BIO * 0.03 * Lrel[d] + rng.normal(0, 0.012)
        toks.append((d + TOD["optical"], MOD["optical"], [ndvi, ndre]))
    # UAV thermal flights
    d = int(rng.integers(0, 4))
    while d < n:
        if rain[d] < 0.5 and rg[d] / 1000. > 12:
            fc = 1 - math.exp(-0.55 * lai[d])
            vp = vpd[d]
            ll = 2.0 - 1.9 * vp
            ul = 4.5
            dT = ll + (1 - rftra[d]) * (ul - ll) + KAPPA_BIO * 1.2 * Lrel[d] + (1 - fc) * (3.0 + rng.normal(0, 1.0)) + rng.normal(0, 0.5)
            toks.append((d + TOD["thermal"], MOD["thermal"], [dT, tmax_[d], vp]))
            d += int(rng.integers(3, 8))
        else:
            d += 1
    # regional spore-trap counts (weekly), informative about inoculum arrival but noisy
    wk = int(rng.integers(0, 7))
    for d in range(wk, n, 7):
        lam = 0.15
        if arrival is not None and d >= arrival - 3:
            lam = 1.5 + 40.0 * float(L[d] + S[d])
        toks.append((d + TOD["trap"], MOD["trap"], [math.log1p(rng.poisson(lam))]))
    for ddate, a in n_events:
        k = (pd.Timestamp(ddate) - days.iloc[0]).days
        if 0 <= k < n:
            toks.append((k + TOD["mgmt"], MOD["mgmt"], [a / 100., 0.0]))
    for ddate, a in irr:
        k = (pd.Timestamp(ddate) - days.iloc[0]).days
        if 0 <= k < n:
            toks.append((k + TOD["mgmt"], MOD["mgmt"], [0.0, a / 3.0]))
    toks.sort(key=lambda x: (x[0], x[1]))
    # landmarks
    stop = T if T is not None else end
    lms = list(range(start, max(start, stop), 3))
    return dict(code=code, year=yr, alt=alt, lat=lat, zone=zone_of(alt, lat), scen=scen, crop=crop, soil=soil,
                n=n, end=end, T=T, J=J, onsets=onsets, vis=vis, landmarks=lms,
                tok_t=np.array([x[0] for x in toks], dtype=np.float32),
                tok_m=np.array([x[1] for x in toks], dtype=np.int64),
                tok_v=[np.array(x[2], dtype=np.float32) for x in toks],
                truth=dict(rftra=rftra.astype(np.float32), ns=ns.astype(np.float32), L=L.astype(np.float32),
                           S=S.astype(np.float32), lai=lai.astype(np.float32), sm=sm.astype(np.float32),
                           pressure=pressure.astype(np.float32), dvs=dvs.astype(np.float32)),
                irr_plan=str(irr_plan), nplan=len(n_events))


def main():
    os.makedirs(OUT, exist_ok=True)
    items = station_years()
    print("station-years", len(items), "stations", len({i[0] for i in items}))
    crop_provider()
    jobs = [(c, y, a, la, lo, w, s) for (c, y, a, la, lo, w) in items for s in range(N_SCEN)]
    with Pool(2) as pool:
        res = pool.map(simulate, jobs, chunksize=8)
    res = [r for r in res if r is not None]
    tag = os.environ.get("TAG", "")
    with open(f"{OUT}/sim_fieldseasons{tag}.pkl", "wb") as f:
        pickle.dump(res, f)
    df = pd.DataFrame([dict(zone=r["zone"], crop=r["crop"], J=r["J"], T=r["T"], year=r["year"],
                            lead=(r["vis"][r["J"]] - r["T"]) if r["J"] >= 0 and r["vis"][r["J"]] is not None else None,
                            ntok=len(r["tok_t"]), nlm=len(r["landmarks"])) for r in res])
    print(len(df)); print(df.J.value_counts()); print(df.groupby("zone").J.value_counts().unstack())
    print(df.groupby("J").lead.describe()); print(df.ntok.describe()); print(df.nlm.sum())


if __name__ == "__main__":
    main()


In [ ]:
%%writefile build_real.py
"""Real-field external benchmark from the FBK soil-moisture dataset (Trentino, three consortia).

Unit = dry-down episode of one tensiometer stream. Competing events: soil-water stress onset (tension
reaching U hPa) versus rewetting (tension returning below L0 after exceeding 200 hPa) before stress.
Asynchronous modalities: gridded weather (daily), tensiometer (daily with gaps), satellite indices on
acquisition days only, irrigation meter events, static soil descriptors."""
import os
import pickle
import re
import numpy as np
import pandas as pd

D = "/content/data/it4lia/"
OUT = "/content/presage/data"
U, L0, MID = 300.0, 150.0, 200.0
STEP = 2


def load(name, c):
    f = f"{D}{name}_consortium{c}.parquet"
    if not os.path.exists(f):
        return None
    df = pd.read_parquet(f)
    if "result_time" not in df.columns:
        df = df.reset_index()
    if "result_time" in df.columns:
        df["result_time"] = pd.to_datetime(df["result_time"])
    return df


def weather(c):
    hw = load("historical_weather_data", c)
    cols = ["temperature_2m_min", "temperature_2m_max", "relative_humidity_2m", "precipitation",
            "direct_radiation", "wind_speed_10m", "et0_fao_evapotranspiration", "soil_moisture_0_to_7cm"]
    w = hw.groupby("result_time")[cols].median().sort_index()
    return w


def acquisitions(g, col):
    """recover satellite acquisition days from a daily, linearly interpolated index series (slope breaks)"""
    v = g[col].values
    d2 = np.abs(np.diff(v, 2))
    idx = np.where(d2 > 1e-6)[0] + 1
    return g.iloc[idx]


def rs_series(c):
    rs = load("remote_sensing_data_final", c)
    if rs is None:
        return {}
    rs = rs.replace([np.inf, -np.inf], np.nan)
    out = {}
    for name, g in rs.groupby("datastream_name"):
        g = g.sort_values("result_time").dropna(subset=["ndvi", "ndmi"])
        a = acquisitions(g, "ndvi")
        out[name] = a[["result_time", "ndvi", "ndmi"]].reset_index(drop=True)
    return out


def sector_of(name):
    m = re.search(r"sector(\d+)", name)
    return m.group(1) if m else None


def build():
    units = []
    for c in range(3):
        W = weather(c)
        sens = load("field_sensor_data", c)
        sens = sens[sens.sensor_type == "Soil Moisture Tension"]
        irr = load("irrigation_data", c)
        soil = load("soil_type_data", c)
        rs = rs_series(c)
        for name, g in sens.groupby("datastream_name"):
            g = g.sort_values("result_time")
            depth = -float(g.ground_offset.iloc[0])
            # --- joins
            rs_k, irr_k, soil_row = [], [], None
            if c == 1:
                rs_k = [k for k in rs if k == name]
                if soil is not None:
                    sr = soil[soil.datastream_name == name]
                    soil_row = sr.iloc[0] if len(sr) else None
            elif c == 0:
                sec = sector_of(name)
                cand = [k for k in rs if sector_of(k) == sec]
                rs_k = sorted(cand, key=lambda k: -len(rs[k]))[:1]  # one representative stream per sector
                irr_k = [k for k in irr.datastream_name.unique() if sector_of(k) == sec]
                if soil is not None:
                    sr = soil[[sector_of(x) == sec for x in soil.datastream_name]]
                    soil_row = sr.iloc[0] if len(sr) else None
            else:
                mm = re.search(r"management(\d)_(\d+)$", name)
                if mm:
                    irr_k = [f"consortium2_Contatore_management{mm.group(1)}_{mm.group(2)}"]
            for yr in sorted(g.result_time.dt.year.unique()):
                t0 = pd.Timestamp(f"{yr}-04-01"); t1 = pd.Timestamp(f"{yr}-10-31")
                gy = g[(g.result_time >= t0) & (g.result_time <= t1)]
                if len(gy) < 30:
                    continue
                day = lambda ts: (ts - t0).days
                toks = []
                st = [depth / 100.]
                if soil_row is not None:
                    st += [float(soil_row["Sand"]) / 100, float(soil_row["Silt"]) / 100, float(soil_row["Clay"]) / 100,
                           float(soil_row["RZD_Modal_(cm)"]) / 100 if not pd.isna(soil_row["RZD_Modal_(cm)"]) else 0.5, 1.0]
                else:
                    st += [0, 0, 0, 0, 0.0]
                toks.append((0.0, 0, st))
                wy = W[(W.index >= t0) & (W.index <= t1)]
                for ts, r in wy.iterrows():
                    toks.append((day(ts) + 0.99, 1, [r.iloc[0], r.iloc[1], r.iloc[2], np.log1p(max(r.iloc[3], 0)),
                                                     r.iloc[4] / 100., r.iloc[5], r.iloc[6], r.iloc[7]]))
                for ts, v in zip(gy.result_time, gy.result.values):
                    if not np.isnan(v):
                        toks.append((day(ts) + 0.5, 2, [np.log1p(max(v, 0)) / np.log(1000.)]))
                for k in rs_k:
                    a = rs[k]; a = a[(a.result_time >= t0) & (a.result_time <= t1)]
                    for ts, nd, nm in zip(a.result_time, a.ndvi, a.ndmi):
                        toks.append((day(ts) + 0.45, 3, [nd, nm]))
                if irr_k:
                    ir = irr[irr.datastream_name.isin(irr_k)]
                    ir = ir[(ir.result_time >= t0) & (ir.result_time <= t1)].groupby("result_time").result.sum()
                    for ts, v in ir.items():
                        if v > 0:
                            toks.append((day(ts) + 0.30, 5, [0.0, float(np.log1p(v))]))
                toks.sort(key=lambda x: (x[0], x[1]))
                # --- episodes (state machine over daily tension)
                vals = gy.set_index(gy.result_time.map(day)).result
                vals = vals[~vals.index.duplicated()]
                days = np.arange(0, (t1 - t0).days + 1)
                ser = vals.reindex(days)
                state = "wet" if (len(vals) and vals.iloc[0] <= L0) else "dry"
                b, peak = (0 if state == "wet" else None), 0.0
                for d in days:
                    x = ser.loc[d]
                    if np.isnan(x):
                        continue
                    if state == "dry":
                        if x <= L0:
                            state, b, peak = "wet", d, x
                        continue
                    peak = max(peak, x)
                    ev = None
                    if x >= U:
                        ev = 0
                    elif peak >= MID and x <= L0:
                        ev = 1
                    if ev is not None:
                        lms = list(range(b, d, STEP))
                        lms = [l for l in lms if not np.isnan(ser.loc[l])] if lms else []
                        if lms:
                            units.append(dict(code=name, cons=c, year=int(yr), T=int(d), J=int(ev), landmarks=lms,
                                              cens=None, **pack(toks)))
                        if ev == 0:
                            state = "dry"
                        else:
                            state, b, peak = "wet", d, x
                if state == "wet" and b is not None:
                    last = int(vals.index.max())
                    lms = [l for l in range(b, last, STEP) if not np.isnan(ser.loc[l])]
                    if lms:
                        units.append(dict(code=name, cons=c, year=int(yr), T=None, J=-1, landmarks=lms,
                                          cens=last, **pack(toks)))
    return units


def pack(toks):
    return dict(tok_t=np.array([x[0] for x in toks], np.float32), tok_m=np.array([x[1] for x in toks], np.int64),
                tok_v=[np.array(x[2], np.float32) for x in toks])


if __name__ == "__main__":
    u = build()
    os.makedirs(OUT, exist_ok=True)
    pickle.dump(u, open(f"{OUT}/real_episodes.pkl", "wb"))
    df = pd.DataFrame([dict(c=x["cons"], J=x["J"], nlm=len(x["landmarks"]), ntok=len(x["tok_t"]),
                            nrs=int((x["tok_m"] == 3).sum()), nirr=int((x["tok_m"] == 5).sum())) for x in u])
    print(len(df)); print(df.groupby("c").J.value_counts().unstack()); print(df.groupby("c")[["nlm", "ntok", "nrs", "nirr"]].mean())
    print(df.nlm.sum())


In [ ]:
%%writefile presage_lib.py
"""PRESAGE: tokenisation, gap-aware selective state-space encoder, physiology-anchored competing-risk head,
self-supervised cross-modal forecasting, and baseline sequence encoders (GRU-D, time-aware Transformer)."""
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

P = 8
K = 3          # causes: 0 water, 1 nitrogen, 2 pathogen
H = 21         # forecast horizon (days)
MODS = 7       # static, weather, soil, optical, thermal, mgmt, spore trap
DIMS = {0: 6, 1: 8, 2: 1, 3: 2, 4: 3, 5: 2, 6: 1}
LINK = [(0, 1.0), (1, -1.0), (2, 1.0)]  # cause j <- (anchor channel, physiological sign)
PROXY_MODE = "sim"


# ----------------------------------------------------------------------------- data
def season_arrays(s):
    T = len(s["tok_t"])
    v = np.zeros((T, P), np.float32)
    for i, x in enumerate(s["tok_v"]):
        v[i, :len(x)] = x
    return s["tok_t"].astype(np.float32), s["tok_m"].astype(np.int64), v


def temp_resp(t, tmin=10., topt=25., tmax=35.):
    t = np.clip(t, tmin, tmax)
    a = (t - tmin) / (topt - tmin); b = (tmax - t) / (tmax - topt)
    return np.clip(a * b ** ((tmax - topt) / (topt - tmin)), 0, 1)


class Prep:
    """Normalisation statistics, CWSI baseline and physics proxies, all fitted on training seasons only."""

    def fit(self, seasons):
        allv = {m: [] for m in range(MODS)}
        th = []
        for s in seasons:
            t, m, v = season_arrays(s)
            for k in range(MODS):
                if (m == k).any():
                    allv[k].append(v[m == k])
            if (m == 4).any():
                th.append(v[m == 4])
        self.mu = np.zeros((MODS, P), np.float32); self.sd = np.ones((MODS, P), np.float32)
        for k in range(MODS):
            if allv[k]:
                a = np.concatenate(allv[k])
                self.mu[k] = a.mean(0); self.sd[k] = a.std(0) + 1e-3
        self.mu[0] = 0; self.sd[0] = 1  # static kept raw
        # non-water-stressed baseline dT = a + b VPD from the lower 10% envelope (practitioner procedure)
        if not th:
            self.a_ll, self.b_ll, self.ul = 0.0, 0.0, 1.0
            return self
        th = np.concatenate(th)
        dT, vpd = th[:, 0], th[:, 2]
        bins = np.quantile(vpd, np.linspace(0, 1, 11))
        xs, ys = [], []
        for lo, hi in zip(bins[:-1], bins[1:]):
            sel = (vpd >= lo) & (vpd <= hi)
            if sel.sum() > 20:
                xs.append(vpd[sel].mean()); ys.append(np.quantile(dT[sel], 0.10))
        self.b_ll, self.a_ll = np.polyfit(xs, ys, 1)
        self.ul = float(np.quantile(dT, 0.90))
        return self

    def proxies(self, t, m, v):
        """physics proxies per token: [CWSI, chlorophyll ratio, 7-day infection pressure]; nan when absent"""
        prox = np.full((len(t), 3), np.nan, np.float32)
        if PROXY_MODE == "real":
            # [7-day atmospheric water deficit ET0 - P (mm/d), canopy moisture index NDMI, unused]
            we = np.where(m == 1)[0]
            if len(we):
                d = v[we, 6] - np.expm1(v[we, 3]); day = np.floor(t[we]); cum = np.cumsum(d)
                for i, dd in enumerate(day):
                    j = np.searchsorted(day, dd - 6, side="left")
                    prox[we[i], 0] = (cum[i] - (cum[j - 1] if j > 0 else 0)) / 7.0
            op = m == 3
            prox[op, 1] = v[op, 1]
            return prox
        th = m == 4
        ll = self.a_ll + self.b_ll * v[th, 2]
        prox[th, 0] = np.clip((v[th, 0] - ll) / np.maximum(self.ul - ll, 0.5), -0.3, 1.5)
        op = m == 3
        prox[op, 1] = np.clip(v[op, 1] / np.maximum(v[op, 0], 0.1), 0, 1.2)
        we = np.where(m == 1)[0]
        if len(we):
            tav = 0.5 * (v[we, 0] + v[we, 1]); rh = v[we, 2]; rain = np.expm1(v[we, 3])
            wet = np.where(rain >= 1.0, 1.0, np.clip((rh - 75.) / 15., 0, 1) * 0.6)
            p = temp_resp(tav) * wet
            day = np.floor(t[we])
            cum = np.cumsum(p)
            for i, d in enumerate(day):
                j = np.searchsorted(day, d - 6, side="left")
                prox[we[i], 2] = (cum[i] - (cum[j - 1] if j > 0 else 0)) / 7.0
        return prox

    def encode(self, s):
        t, m, v = season_arrays(s)
        prox = self.proxies(t, m, v)
        vn = (v - self.mu[m]) / self.sd[m]
        mask = np.zeros((len(t), P), np.float32)
        for k, d in DIMS.items():
            mask[m == k, :d] = 1
        vn = vn * mask
        return dict(t=t, m=m, v=vn, vmask=mask, prox=prox)


def landmark_targets(s):
    lms = np.array(s["landmarks"], np.int64)
    if s["T"] is None and s.get("cens") is not None:   # right-censored (end of record)
        delta = s["cens"] - lms; cause = np.full(len(lms), -2, np.int64)
    elif s["T"] is None:                                # season completed without onset
        delta = np.full(len(lms), 10 ** 4, np.int64); cause = np.full(len(lms), -1, np.int64)
    else:
        delta = s["T"] - lms; cause = np.full(len(lms), s["J"], np.int64)
    return lms, delta, cause


# ----------------------------------------------------------------------------- batching
def collate(encs, seasons, lm_sel=None, drop_mod=None, token_drop=0.0, window_drop=0.0, rng=None,
            n_fc=48, max_lb=7):
    """Pads token streams; optionally removes tokens (augmentation / robustness tests) and builds
    cross-modal forecasting targets from removed and retained tokens."""
    rng = rng or np.random.default_rng(0)
    items = []
    for e, s in zip(encs, seasons):
        t, m, v, vm, pr = e["t"], e["m"], e["v"], e["vmask"], e["prox"]
        keep = np.ones(len(t), bool)
        if drop_mod is not None:
            for dm in np.atleast_1d(drop_mod):
                keep &= m != dm
        if token_drop > 0:
            keep &= ~((rng.random(len(t)) < token_drop) & (m != 0))
        if window_drop > 0 and rng.random() < window_drop:
            mod = int(rng.choice([2, 3, 4, 6]))
            a = rng.uniform(0, max(1, t.max() - 30)); w = rng.uniform(15, 40)
            keep &= ~((m == mod) & (t >= a) & (t <= a + w))
        kt, km, kv, kvm, kpr = t[keep], m[keep], v[keep], vm[keep], pr[keep]
        # forecasting targets: any non-static/non-mgmt token (removed tokens favoured)
        cand = np.where(((m >= 1) & (m <= 4)) | (m == 6))[0]
        rem = cand[~keep[cand]]
        nrem = min(len(rem), n_fc // 2)
        pick = list(rng.choice(rem, nrem, replace=False)) if nrem else []
        pick += list(rng.choice(cand, min(len(cand), n_fc - nrem), replace=False))
        pick = np.array(pick, np.int64)
        lb = rng.uniform(0, max_lb, len(pick)).astype(np.float32)
        src = np.searchsorted(kt, t[pick] - lb, side="left") - 1  # last retained token strictly before t - lb
        ok = src >= 0
        pick, src = pick[ok], src[ok]
        lms, delta, cause = landmark_targets(s)
        if lm_sel is not None:
            sel = lm_sel(lms)
            lms, delta, cause = lms[sel], delta[sel], cause[sel]
        lidx = np.searchsorted(kt, lms + 1.0, side="left") - 1
        lastm = np.full((len(lms), MODS), -1, np.int64)
        for mm_ in range(1, MODS):
            pos = np.where(km == mm_)[0]
            if len(pos):
                j = np.searchsorted(kt[pos], lms + 1.0, side="left") - 1
                lastm[:, mm_] = np.where(j >= 0, pos[np.maximum(j, 0)], -1)
        items.append(dict(lastm=lastm, t=kt, m=km, v=kv, prox=kpr, fc_src=src, fc_m=m[pick], fc_v=v[pick], fc_vm=vm[pick],
                          fc_dt=t[pick] - kt[src], lidx=np.maximum(lidx, 0), lms=lms, delta=delta, cause=cause))
    B = len(items); T = max(len(x["t"]) for x in items)
    tt = np.zeros((B, T), np.float32); mm = np.zeros((B, T), np.int64); vv = np.zeros((B, T, P), np.float32)
    pad = np.zeros((B, T), bool); prox = np.full((B, T, 3), np.nan, np.float32)
    for i, x in enumerate(items):
        n = len(x["t"]); tt[i, :n] = x["t"]; mm[i, :n] = x["m"]; vv[i, :n] = x["v"]; pad[i, n:] = True
        prox[i, :n] = x["prox"]
        if n < T:
            tt[i, n:] = x["t"][-1]
    dt = np.diff(tt, axis=1, prepend=tt[:, :1]); dt = np.clip(dt, 0, None)
    out = dict(t=torch.tensor(tt), m=torch.tensor(mm), v=torch.tensor(vv), pad=torch.tensor(pad),
               dt=torch.tensor(dt), prox=torch.tensor(prox))
    # flattened landmark and forecasting indices
    lb_b, lb_i, dl, cs, lm_day, lastm = [], [], [], [], [], []
    fb, fi, fm, fv, fvm, fdt = [], [], [], [], [], []
    for i, x in enumerate(items):
        lb_b += [i] * len(x["lidx"]); lb_i += list(x["lidx"]); dl += list(x["delta"]); cs += list(x["cause"])
        lm_day += list(x["lms"]); lastm.append(x["lastm"])
        fb += [i] * len(x["fc_src"]); fi += list(x["fc_src"]); fm += list(x["fc_m"]); fv.append(x["fc_v"])
        fvm.append(x["fc_vm"]); fdt += list(x["fc_dt"])
    out.update(lb_b=torch.tensor(lb_b, dtype=torch.long), lb_i=torch.tensor(lb_i, dtype=torch.long),
               delta=torch.tensor(dl, dtype=torch.long), cause=torch.tensor(cs, dtype=torch.long),
               lm_day=torch.tensor(lm_day, dtype=torch.float32),
               lastm=torch.tensor(np.concatenate(lastm) if lastm else np.zeros((0, MODS)), dtype=torch.long),
               fb=torch.tensor(fb, dtype=torch.long), fi=torch.tensor(fi, dtype=torch.long),
               fm=torch.tensor(fm, dtype=torch.long),
               fv=torch.tensor(np.concatenate(fv) if fv else np.zeros((0, P)), dtype=torch.float32),
               fvm=torch.tensor(np.concatenate(fvm) if fvm else np.zeros((0, P)), dtype=torch.float32),
               fdt=torch.tensor(fdt, dtype=torch.float32))
    return out


# ----------------------------------------------------------------------------- encoders
def fourier(x, n=8, maxp=200.0):
    freqs = torch.exp(torch.linspace(math.log(1.0), math.log(maxp), n, device=x.device))
    a = x[..., None] * (2 * math.pi / freqs)
    return torch.cat([torch.sin(a), torch.cos(a)], -1)


class TokenEmbed(nn.Module):
    def __init__(self, D):
        super().__init__()
        self.proj = nn.ModuleList([nn.Linear(P, D) for _ in range(MODS)])
        self.memb = nn.Embedding(MODS, D)
        self.temb = nn.Linear(32, D)
        self.ln = nn.LayerNorm(D)

    def forward(self, b):
        v, m, t, dt = b["v"], b["m"], b["t"], b["dt"]
        x = torch.zeros(*v.shape[:2], self.memb.embedding_dim)
        for k in range(MODS):
            sel = m == k
            if sel.any():
                x[sel] = self.proj[k](v[sel])
        tf = torch.cat([fourier(t), fourier(torch.log1p(dt), maxp=8.0)], -1)
        return self.ln(x + self.memb(m) + self.temb(tf))


def seq_scan(log_a, u):
    """sequential form of the same recurrence (faster on CPU for short sequences)"""
    a = torch.exp(log_a).unbind(1)
    uu = u.unbind(1)
    B, T, D, N = u.shape
    h = torch.zeros(B, D, N); hs = []
    for t in range(T):
        h = a[t] * h + uu[t]
        hs.append(h)
    return torch.stack(hs, 1)


def chunk_scan(log_a, u, Q=16):
    """h_t = exp(log_a_t) * h_{t-1} + u_t over dim 1, computed chunk-wise with stable segment sums.
    log_a, u: (B, T, D, N)"""
    B, T, D, N = u.shape
    pad = (-T) % Q
    if pad:
        log_a = F.pad(log_a, (0, 0, 0, 0, 0, pad)); u = F.pad(u, (0, 0, 0, 0, 0, pad))
    C = log_a.shape[1] // Q
    la = log_a.view(B, C, Q, D, N).permute(0, 1, 3, 4, 2)  # B C D N Q
    uu = u.view(B, C, Q, D, N).permute(0, 1, 3, 4, 2)
    tri = torch.tril(torch.ones(Q, Q, dtype=torch.bool))
    h = torch.zeros(B, D, N)
    outs = []
    for c in range(C):
        cum = torch.cumsum(la[:, c], -1)  # B D N Q
        seg = cum[..., :, None] - cum[..., None, :]
        seg = seg.masked_fill(~tri, float("-inf"))
        Lm = torch.exp(seg)
        hc = torch.einsum("bdnts,bdns->bdnt", Lm, uu[:, c]) + torch.exp(cum) * h[..., None]
        outs.append(hc)
        h = hc[..., -1]
    hs = torch.stack(outs, 1).permute(0, 1, 4, 2, 3).reshape(B, C * Q, D, N)
    return hs[:, :T]


class GapSSM(nn.Module):
    """Selective diagonal state-space layer whose discretisation uses the real elapsed time between tokens.
    gap_aware=False replaces elapsed time by a unit step (ordinal sequence); selective=False fixes Delta."""

    def __init__(self, D, N=8, gap_aware=True, selective=True, expand=2):
        super().__init__()
        E = expand * D
        self.inp = nn.Linear(D, 2 * E)
        self.dlt = nn.Linear(E, E)
        self.Bp = nn.Linear(E, N); self.Cp = nn.Linear(E, N)
        tau = torch.exp(torch.linspace(math.log(0.7), math.log(60.0), N))  # memory time constants (days)
        self.log_lam = nn.Parameter(torch.log(1.0 / tau).repeat(E, 1))
        self.Dskip = nn.Parameter(torch.ones(E))
        self.out = nn.Linear(E, D)
        self.ln = nn.LayerNorm(D)
        self.gap_aware, self.selective = gap_aware, selective
        self.fixed_delta = nn.Parameter(torch.zeros(E))

    def forward(self, x, dt, pad):
        r = x
        xz = self.inp(self.ln(x)); xi, z = xz.chunk(2, -1)
        xi = F.silu(xi)
        delta = F.softplus(self.dlt(xi) if self.selective else self.fixed_delta.expand_as(xi)) + 1e-3
        lam = torch.exp(self.log_lam)  # E N
        gap = dt if self.gap_aware else (dt > 0).float() + 0.0 * dt
        log_a = -lam * (gap[..., None, None] + delta[..., None])
        beta = (1 - torch.exp(-lam * delta[..., None])) / lam  # zero-order-hold input gain
        u = beta * xi[..., None] * self.Bp(xi)[:, :, None, :]
        pm = pad[..., None, None]
        log_a = log_a.masked_fill(pm, 0.0); u = u.masked_fill(pm, 0.0)
        h = seq_scan(log_a, u)
        y = (h * self.Cp(xi)[:, :, None, :]).sum(-1) + self.Dskip * xi
        y = y * F.silu(z)
        return r + self.out(y)


class GapSSMEncoder(nn.Module):
    def __init__(self, D=48, N=8, L=2, gap_aware=True, selective=True):
        super().__init__()
        self.emb = TokenEmbed(D)
        self.layers = nn.ModuleList([GapSSM(D, N, gap_aware, selective) for _ in range(L)])
        self.lnf = nn.LayerNorm(D)

    def forward(self, b):
        x = self.emb(b)
        for l in self.layers:
            x = l(x, b["dt"], b["pad"])
        return self.lnf(x)


class TimeTransformerEncoder(nn.Module):
    """Causal Transformer over the same tokens with continuous-time embeddings (attention baseline)."""

    def __init__(self, D=48, L=2, heads=4):
        super().__init__()
        self.emb = TokenEmbed(D)
        layer = nn.TransformerEncoderLayer(D, heads, 2 * D, dropout=0.1, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, L)
        self.lnf = nn.LayerNorm(D)

    def forward(self, b):
        x = self.emb(b)
        T = x.shape[1]
        causal = torch.triu(torch.ones(T, T, dtype=torch.bool), 1)
        # tokens may attend to tokens with equal or earlier time stamps only (no future leakage)
        y = self.enc(x, mask=causal, src_key_padding_mask=b["pad"])
        return self.lnf(torch.nan_to_num(y))


class GRUDEncoder(nn.Module):
    """GRU-D style recurrent baseline: per-token update with input decay toward the empirical mean and
    hidden-state decay as a function of elapsed time."""

    def __init__(self, D=48):
        super().__init__()
        self.emb = TokenEmbed(D)
        self.cell = nn.GRUCell(D, D)
        self.gamma_h = nn.Linear(1, D)
        self.lnf = nn.LayerNorm(D)

    def forward(self, b):
        x = self.emb(b)
        B, T, D = x.shape
        h = torch.zeros(B, D); outs = []
        G = torch.exp(-F.relu(self.gamma_h(b["dt"][..., None]))).unbind(1)
        X = x.unbind(1); PADS = b["pad"][..., None].unbind(1)
        for i in range(T):
            hn = self.cell(X[i], h * G[i])
            keep = PADS[i]
            h = torch.where(keep, h, hn)
            outs.append(h)
        return self.lnf(torch.stack(outs, 1))


# ----------------------------------------------------------------------------- heads and model
class CompetingRiskHead(nn.Module):
    def __init__(self, D, anchor=True, ctx=0):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(D + ctx, 2 * D), nn.GELU(), nn.Linear(2 * D, H * (K + 1)))
        self.anchor = anchor
        self.K = K
        self.gamma = nn.Parameter(torch.zeros(K))
        self.chan = [c for c, _ in LINK[:K]]
        self.sign = torch.tensor([s for _, s in LINK[:K]])

    def forward(self, h, z=None):
        lg = self.mlp(h).view(-1, H, self.K + 1)
        if self.anchor and z is not None:
            zz = torch.stack([z[:, c] if c is not None else torch.zeros_like(z[:, 0]) for c in self.chan], 1)
            link = self.sign * F.softplus(self.gamma) * zz  # (n, K)
            lg = lg + F.pad(link, (1, 0))[:, None, :]
        return lg


class PRESAGE(nn.Module):
    def __init__(self, D=48, N=8, L=2, encoder="ssm", anchor=True, gap_aware=True, selective=True, lastctx=False):
        super().__init__()
        self.lastctx = lastctx
        if encoder == "ssm":
            self.enc = GapSSMEncoder(D, N, L, gap_aware, selective)
        elif encoder == "transformer":
            self.enc = TimeTransformerEncoder(D, L)
        elif encoder == "grud":
            self.enc = GRUDEncoder(D)
        self.anchor = anchor
        self.zhead = nn.Linear(D, 3)
        self.cr = CompetingRiskHead(D, anchor, ctx=6 + 16 + ((MODS - 1) * (P + 2) if lastctx else 0))
        self.fc = nn.Sequential(nn.Linear(D + D + 16, 2 * D), nn.GELU(), nn.Linear(2 * D, P))
        self.fmemb = nn.Embedding(MODS, D)

    def forward(self, b):
        y = self.enc(b)
        hl = y[b["lb_b"], b["lb_i"]]
        z = self.zhead(hl)
        ctx = torch.cat([b["v"][b["lb_b"], 0, :6], fourier(b["lm_day"], 8, 200.0)], -1)  # static token + landmark time
        if self.lastctx:  # last observed value, presence and log-age of every modality at the landmark
            li = b["lastm"][:, 1:]; ok = (li >= 0).float()
            bb = b["lb_b"][:, None].expand_as(li)
            vals = b["v"][bb, li.clamp(min=0)] * ok[..., None]
            age = torch.log1p((b["lm_day"][:, None] + 1.0 - b["t"][bb, li.clamp(min=0)]).clamp(min=0)) * ok
            ctx = torch.cat([ctx, vals.flatten(1), ok, age], -1)
        logits = self.cr(torch.cat([hl, ctx], -1), z if self.anchor else None)
        return y, logits, z

    def forecast_loss(self, b, y):
        if len(b["fb"]) == 0:
            return torch.tensor(0.0)
        hs = y[b["fb"], b["fi"]]
        q = torch.cat([hs, self.fmemb(b["fm"]), fourier(b["fdt"], 8, 30.0)], -1)
        pred = self.fc(q)
        return (((pred - b["fv"]) ** 2) * b["fvm"]).sum() / b["fvm"].sum().clamp(min=1)

    def anchor_loss(self, b, y):
        """predictive anchoring: z read from the state just before a token must match that token's proxy"""
        z = self.zhead(y[:, :-1])
        tgt = b["prox"][:, 1:]
        ok = ~torch.isnan(tgt) & ~b["pad"][:, 1:, None]
        if ok.sum() == 0:
            return torch.tensor(0.0)
        return F.smooth_l1_loss(z[ok], torch.nan_to_num(tgt)[ok])


def cr_nll(logits, delta, cause):
    """discrete-time competing-risk negative log-likelihood; delta in days (>=1), cause -1 = no event"""
    logp = F.log_softmax(logits, -1)  # n H K+1
    n = logits.shape[0]
    hgrid = torch.arange(1, H + 1)[None, :]
    ev = (cause >= 0) & (delta <= H)
    surv_steps = torch.where(ev, delta - 1, torch.full_like(delta, H))
    surv_steps = torch.where(cause == -2, delta.clamp(max=H), surv_steps)  # censored before the horizon
    sm = (hgrid <= surv_steps[:, None]).float()
    ll = (logp[:, :, 0] * sm).sum(1)
    idx = (delta.clamp(1, H) - 1)
    ll_ev = logp[torch.arange(n), idx, (cause + 1).clamp(min=0)]
    ll = ll + torch.where(ev, ll_ev, torch.zeros_like(ll_ev))
    return -ll.mean()


def cif_from_logits(logits):
    p = torch.softmax(logits, -1)
    surv = torch.cumprod(p[:, :, 0], 1)
    prev = torch.cat([torch.ones_like(surv[:, :1]), surv[:, :-1]], 1)
    inc = prev[:, :, None] * p[:, :, 1:]
    return torch.cumsum(inc, 1)  # n H K


def distill_loss(student_logits, teacher_logits, T=1.0):
    return F.kl_div(F.log_softmax(student_logits / T, -1), F.softmax(teacher_logits / T, -1),
                    reduction="batchmean") / H


In [ ]:
%%writefile train.py
"""Training / inference utilities for PRESAGE and the sequence baselines."""
import copy
import time
import numpy as np
import torch
from presage_lib import (PRESAGE, Prep, collate, cr_nll, cif_from_logits, distill_loss, H, K)

torch.set_num_threads(2)


def batches(idx, enc, seasons, bs, rng, shuffle=True, **kw):
    idx = np.array(idx)
    if shuffle:
        idx = idx[rng.permutation(len(idx))]
    # length bucketing for speed
    chunks = [idx[i:i + bs * 8] for i in range(0, len(idx), bs * 8)]
    out = []
    for c in chunks:
        c = sorted(c, key=lambda i: len(enc[i]["t"]))
        out += [c[i:i + bs] for i in range(0, len(c), bs)]
    if shuffle:
        rng.shuffle(out)
    for b in out:
        yield b, collate([enc[i] for i in b], [seasons[i] for i in b], rng=rng, **kw)


def pretrain(model, tr, enc, seasons, epochs=6, bs=32, lr=2e-3, seed=0, log=print):
    rng = np.random.default_rng(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    for ep in range(epochs):
        tot, nb, t0 = 0.0, 0, time.time()
        model.train()
        for _, b in batches(tr, enc, seasons, bs, rng, window_drop=0.8, token_drop=0.1):
            y = model.enc(b)
            loss = model.forecast_loss(b, y) + (0.5 * model.anchor_loss(b, y) if model.anchor else 0.0)
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tot += loss.item(); nb += 1
        log(f"  pre ep{ep} loss {tot / nb:.4f} {time.time() - t0:.0f}s")
    return model


def evaluate_nll(model, idx, enc, seasons, bs=64):
    model.eval(); tot, n = 0.0, 0
    rng = np.random.default_rng(123)
    with torch.no_grad():
        for _, b in batches(idx, enc, seasons, bs, rng, shuffle=False):
            _, lg, _ = model(b)
            if len(lg):
                tot += cr_nll(lg, b["delta"], b["cause"]).item() * len(lg); n += len(lg)
    return tot / max(n, 1)


def finetune(model, tr, va, enc, seasons, epochs=14, bs=32, lr=1.5e-3, seed=0, aux=True, patience=4,
             teacher=None, log=print):
    rng = np.random.default_rng(seed + 7)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    best, best_state, bad = 1e9, None, 0
    for ep in range(epochs):
        model.train(); t0 = time.time(); tot, nb = 0.0, 0
        for bidx, b in batches(tr, enc, seasons, bs, rng, window_drop=0.3, token_drop=0.05):
            y, lg, z = model(b)
            loss = cr_nll(lg, b["delta"], b["cause"])
            if aux:
                loss = loss + 0.2 * model.forecast_loss(b, y)
                if model.anchor:
                    loss = loss + 0.2 * model.anchor_loss(b, y)
            if teacher is not None:
                with torch.no_grad():
                    _, tlg, _ = teacher(b)
                loss = loss + 1.0 * distill_loss(lg, tlg)
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tot += loss.item(); nb += 1
        sched.step()
        v = evaluate_nll(model, va, enc, seasons)
        log(f"  ft ep{ep} train {tot / nb:.4f} val {v:.4f} {time.time() - t0:.0f}s")
        if v < best - 1e-4:
            best, best_state, bad = v, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    return model, best


def predict(model, idx, enc, seasons, bs=64, **kw):
    """returns per-landmark records: season index, landmark day, delta, cause, CIF (H,K), z"""
    model.eval(); rng = np.random.default_rng(99)
    S, LM, D, C, CIF, Z = [], [], [], [], [], []
    with torch.no_grad():
        for bidx, b in batches(idx, enc, seasons, bs, rng, shuffle=False, **kw):
            _, lg, z = model(b)
            cif = cif_from_logits(lg).numpy()
            sidx = np.array(bidx)[b["lb_b"].numpy()]
            S.append(sidx); LM.append(b["lm_day"].numpy()); D.append(b["delta"].numpy()); C.append(b["cause"].numpy())
            CIF.append(cif); Z.append(z.numpy())
    return dict(season=np.concatenate(S), lm=np.concatenate(LM), delta=np.concatenate(D),
                cause=np.concatenate(C), cif=np.concatenate(CIF), z=np.concatenate(Z))


In [ ]:
%%writefile splits.py
import numpy as np


def temporal_split(S):
    y = np.array([s["year"] for s in S])
    return dict(train=np.where(y <= 2010)[0], val=np.where((y >= 2011) & (y <= 2013))[0],
                cal=np.where((y >= 2014) & (y <= 2017))[0], test=np.where(y >= 2018)[0])


def lozo_split(S, zone, seed=0):
    """leave-one-zone-out: test = all seasons of the held-out zone; remaining zones split by station
    into train / val / calibration so that no station contributes to two partitions"""
    rng = np.random.default_rng(100 + zone)
    z = np.array([s["zone"] for s in S]); st = np.array([s["code"] for s in S])
    test = np.where(z == zone)[0]
    rest = np.where(z != zone)[0]
    stations = np.unique(st[rest]); rng.shuffle(stations)
    n = len(stations); nv = max(2, int(0.15 * n)); nc = max(3, int(0.2 * n))
    vs, cs = set(stations[:nv]), set(stations[nv:nv + nc])
    val = np.array([i for i in rest if st[i] in vs]); cal = np.array([i for i in rest if st[i] in cs])
    train = np.array([i for i in rest if st[i] not in vs and st[i] not in cs])
    return dict(train=train, val=val, cal=cal, test=test)


In [ ]:
%%writefile eval_lib.py
"""Metrics, conformal lead-time lower bounds, weighted conformal under shift and conformal risk control."""
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

H, K, W = 21, 3, 14


def cindex_cr(risk, delta, cause, j, horizon=H):
    """truncated cause-specific concordance for competing risks (Wolbers-type comparability)"""
    ev = (cause == j) & (delta <= horizon)
    idx = np.where(ev)[0]
    if len(idx) == 0:
        return np.nan
    num = den = 0.0
    for i in idx:
        comp = (delta > delta[i]) | ((cause != j) & (cause >= 0) & (delta <= delta[i]))
        comp[i] = False
        r = risk[comp]
        den += len(r)
        num += (risk[i] > r).sum() + 0.5 * (risk[i] == r).sum()
    return num / den if den else np.nan


def brier_ibs(cif, delta, cause):
    """integrated (over 1..H) cause-specific Brier score; averaged over causes"""
    out = []
    keep = ~((cause == -2) & (delta < H))
    cif, delta, cause = cif[keep], delta[keep], cause[keep]
    for j in range(cif.shape[2]):
        bs = []
        for h in range(1, H + 1):
            y = ((cause == j) & (delta <= h)).astype(float)
            bs.append(np.mean((y - cif[:, h - 1, j]) ** 2))
        out.append(np.mean(bs))
    return float(np.mean(out)), out


def ece_binary(p, y, bins=10):
    edges = np.linspace(0, 1, bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        s = (p >= lo) & (p < hi if hi < 1 else p <= hi)
        if s.any():
            e += s.mean() * abs(p[s].mean() - y[s].mean())
    return e


def point_metrics(pred, window=W):
    cif, d, c = pred["cif"], pred["delta"], pred["cause"]
    cidx = [cindex_cr(cif[:, H - 1, j], d, c, j) for j in range(cif.shape[2])]
    ibs, _ = brier_ibs(cif, d, c)
    ev = (c >= 0) & (d <= H)
    f1 = f1_score(c[ev], cif[ev, H - 1].argmax(1), average="macro") if ev.any() else np.nan
    pw = cif[:, window - 1].sum(1); yw = ((c >= 0) & (d <= window)).astype(float)
    return dict(cindex=float(np.nanmean(cidx)), cindex_by_cause=[float(x) for x in cidx], ibs=ibs,
                f1=float(f1), ece=float(ece_binary(pw, yw)))


# --------------------------------------------------------------------------- alerts + CRC
def season_alert_table(pred, seasons, risk, theta, window=W, targets=(0, 1, 2)):
    """per-season: hit (alert within [T-window, T)), first alert day in window, false alert count."""
    out = {}
    tg = np.array(targets)
    for s in np.unique(pred["season"]):
        sel = pred["season"] == s
        lm = pred["lm"][sel]; r = risk[sel]; d = pred["delta"][sel]; c = pred["cause"][sel]
        alert = r >= theta
        ev = bool(np.isin(c[0], tg))
        T = lm[0] + d[0] if ev else None
        inwin = (d <= window) & np.isin(c, tg)
        hit = bool((alert & inwin).any()) if ev else None
        first = lm[alert & inwin].min() if ev and hit else None
        fa = int((alert & ~inwin).sum())
        out[int(s)] = dict(ev=ev, T=T, hit=hit, first=first, fa=fa, n=int(sel.sum()), alerts=int(alert.sum()),
                           has_window=bool(inwin.any()))
    return out


def crc_threshold(pred, seasons, risk, alpha, grid=None, window=W, targets=(0, 1, 2)):
    """conformal risk control: largest threshold whose (n R + 1)/(n + 1) bound on missed onsets is <= alpha"""
    grid = np.unique(np.quantile(risk, np.linspace(0, 1, 400))) if grid is None else grid
    sid = np.unique(pred["season"]); n = len(sid)
    best = grid.min()
    for th in np.sort(grid)[::-1]:
        tab = season_alert_table(pred, seasons, risk, th, window, targets)
        miss = np.mean([1.0 if (v["ev"] and v["has_window"] and not v["hit"]) else 0.0 for v in tab.values()])
        if (n * miss + 1) / (n + 1) <= alpha:
            best = th
            break
    return float(best)


def alert_metrics(pred, seasons, risk, theta, cause_pred=None, window=W, targets=(0, 1, 2)):
    tab = season_alert_table(pred, seasons, risk, theta, window, targets)
    evs = [s for s, v in tab.items() if v["ev"] and v["has_window"]]
    miss = np.mean([0.0 if tab[s]["hit"] else 1.0 for s in evs])
    leads_on, leads_vis, cause_ok = [], [], []
    for s in evs:
        v = tab[s]
        if not v["hit"]:
            continue
        S = seasons[s]; J = S["J"]
        leads_on.append(v["T"] - v["first"])
        if "vis" in S and S["vis"][J] is not None:
            leads_vis.append(S["vis"][J] - v["first"])
        if cause_pred is not None and len(targets) > 1:
            sel = (pred["season"] == s) & (pred["lm"] == v["first"])
            cause_ok.append((J, int(cause_pred[sel][0])))
    tot_alerts = sum(v["alerts"] for v in tab.values()); tot_fa = sum(v["fa"] for v in tab.values())
    tot_lm = sum(v["n"] for v in tab.values())
    res = dict(miss=float(miss), lead_onset=float(np.median(leads_on)) if leads_on else np.nan,
               lead_visible=float(np.median(leads_vis)) if leads_vis else np.nan,
               lead_visible_mean=float(np.mean(leads_vis)) if leads_vis else np.nan,
               false_alert_frac=float(tot_fa / max(tot_alerts, 1)), fa_per_100lm=float(100 * tot_fa / tot_lm),
               alert_rate=float(tot_alerts / tot_lm))
    if cause_ok:
        a = np.array(cause_ok)
        res["alert_cause_f1"] = float(f1_score(a[:, 0], a[:, 1], average="macro"))
    return res, leads_vis


# --------------------------------------------------------------------------- lead-time lower bounds
def base_lower(cif, a0=0.1):
    """smallest horizon at which the any-cause CIF reaches a0 (H+1 when never reached)"""
    tot = cif.sum(2)
    hit = tot >= a0
    first = np.where(hit.any(1), hit.argmax(1) + 1, H + 1)
    return first.astype(float)


def one_per_season(pred, rng):
    idx = []
    for s in np.unique(pred["season"]):
        w = np.where(pred["season"] == s)[0]
        idx.append(rng.choice(w))
    return np.array(idx)


def weighted_quantile(scores, w_cal, w_test, q):
    """Tibshirani et al. weighted conformal quantile with the test point's weight placed at +inf"""
    o = np.argsort(scores); s = scores[o]; wc = w_cal[o]
    tot = wc.sum() + w_test
    cdf = np.cumsum(wc) / tot
    k = np.searchsorted(cdf, q, side="left")
    return s[k] if k < len(s) else np.inf


def lpb_calibrate(cal, alpha=0.1, a0=0.1, seed=0, n_rep=20):
    """returns the split-conformal correction (median over repeated one-landmark-per-season draws)"""
    rng = np.random.default_rng(seed)
    qs = []
    for _ in range(n_rep):
        idx = one_per_season(cal, rng)
        q = base_lower(cal["cif"][idx], a0)
        y = np.minimum(cal["delta"][idx], H + 1).astype(float)
        sc = q - y
        n = len(sc); k = int(np.ceil((n + 1) * (1 - alpha)))
        qs.append(np.sort(sc)[min(k, n) - 1])
    return float(np.median(qs))


def lpb_weighted(cal, test, f_cal, f_test, alpha=0.1, a0=0.1, seed=0, n_rep=10):
    """covariate-shift weighted conformal lower bounds; density ratio from a probabilistic domain classifier"""
    rng = np.random.default_rng(seed)
    X = np.vstack([f_cal, f_test]); yd = np.r_[np.zeros(len(f_cal)), np.ones(len(f_test))]
    clf = LogisticRegression(max_iter=2000, C=0.5).fit(X, yd)
    pc = np.clip(clf.predict_proba(f_cal)[:, 1], 1e-3, 1 - 1e-3); pt = np.clip(clf.predict_proba(f_test)[:, 1], 1e-3, 1 - 1e-3)
    wc_all = pc / (1 - pc); wt_all = pt / (1 - pt)
    cap = 20.0 * np.median(wc_all)  # truncated density ratios (limits the influence of non-overlapping regions)
    wc_all = np.minimum(wc_all, cap); wt_all = np.minimum(wt_all, cap)
    qb_test = base_lower(test["cif"], a0)
    corr = np.zeros((n_rep, len(f_test)))
    for r in range(n_rep):
        idx = one_per_season(cal, rng)
        sc = base_lower(cal["cif"][idx], a0) - np.minimum(cal["delta"][idx], H + 1)
        wc = wc_all[idx]
        o = np.argsort(sc); s = sc[o]; w = wc[o]; cw = np.cumsum(w)
        tot = cw[-1] + wt_all
        # per test point quantile: smallest s with cumulative weight >= (1-alpha) * total
        need = (1 - alpha) * tot
        k = np.searchsorted(cw, need, side="left")
        corr[r] = np.where(k < len(s), s[np.minimum(k, len(s) - 1)], np.inf)
    c = np.median(corr, 0)
    return qb_test - c, wt_all


def lpb_eval(lpb, test):
    y = np.minimum(test["delta"], 10 ** 4)
    known = ~((test["cause"] == -2) & (y < lpb))  # censored before the bound: coverage unknown
    cov = float(np.mean((y >= lpb)[known]))
    ev = test["cause"] >= 0
    cov_ev = float(np.mean(y[ev & (test["delta"] <= H)] >= lpb[ev & (test["delta"] <= H)]))
    finite = np.clip(lpb, 0, H + 1)
    return dict(coverage=cov, coverage_events=cov_ev, mean_lpb=float(np.mean(finite)),
                informative=float(np.mean(finite >= 3)))


In [ ]:
%%writefile baselines_tab.py
"""Landmark summary features and tabular / rule baselines."""
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.ensemble import HistGradientBoostingClassifier
from presage_lib import season_arrays, Prep, H, K, CompetingRiskHead, cr_nll, cif_from_logits

FEATS = ["maize", "SMW", "SMFCF", "SM0", "alt", "lat", "day", "gdd", "tmax7", "rh7", "rain7", "rain30", "vpd7", "rg7",
         "press7", "soil_last", "soil_d5", "soil_age", "ndvi_last", "ndre_last", "ratio_last", "ndvi_max", "ndvi_d",
         "ratio_d", "opt_age", "cwsi_last", "cwsi_mean2", "th_age", "n_tot", "n_age", "irr_tot", "irr_age", "trap_last",
         "trap_sum21"]


def landmark_features(s, prep, drop_mod=None, keep_mask=None):
    t, m, v = season_arrays(s)
    keep = np.ones(len(t), bool) if keep_mask is None else keep_mask.copy()
    if drop_mod is not None:
        for dm in np.atleast_1d(drop_mod):
            keep &= m != dm
    t, m, v = t[keep], m[keep], v[keep]
    prox = prep.proxies(t, m, v)
    rows = []
    for lm in s["landmarks"]:
        c = t < lm + 1.0
        f = dict(zip(FEATS[:6], v[m == 0][0][:6]))
        f["day"] = lm
        w = c & (m == 1)
        wl = v[w]; wt = t[w]; pw = prox[w, 2]
        f["gdd"] = wl[-1, 7] if len(wl) else np.nan
        r7 = wt >= lm - 6; r30 = wt >= lm - 29
        f["tmax7"] = wl[r7, 1].mean() if r7.any() else np.nan
        f["rh7"] = wl[r7, 2].mean() if r7.any() else np.nan
        f["rain7"] = np.expm1(wl[r7, 3]).sum() if r7.any() else np.nan
        f["rain30"] = np.expm1(wl[r30, 3]).sum() if r30.any() else np.nan
        f["vpd7"] = wl[r7, 6].mean() if r7.any() else np.nan
        f["rg7"] = wl[r7, 4].mean() if r7.any() else np.nan
        f["press7"] = pw[-1] if len(pw) else np.nan
        so = c & (m == 2); sv = v[so, 0]; st = t[so]
        f["soil_last"] = sv[-1] if len(sv) else np.nan
        f["soil_d5"] = (sv[-1] - sv[st <= st[-1] - 5][-1]) if len(sv) and (st <= st[-1] - 5).any() else np.nan
        f["soil_age"] = lm + 1 - st[-1] if len(st) else 99
        op = c & (m == 3); ov = v[op]; ot = t[op]; orat = prox[op, 1]
        f["ndvi_last"] = ov[-1, 0] if len(ov) else np.nan
        f["ndre_last"] = ov[-1, 1] if len(ov) else np.nan
        f["ratio_last"] = orat[-1] if len(ov) else np.nan
        f["ndvi_max"] = ov[:, 0].max() if len(ov) else np.nan
        f["ndvi_d"] = ov[-1, 0] - ov[-2, 0] if len(ov) > 1 else np.nan
        f["ratio_d"] = orat[-1] - orat[-2] if len(ov) > 1 else np.nan
        f["opt_age"] = lm + 1 - ot[-1] if len(ot) else 99
        th = c & (m == 4); cw = prox[th, 0]; tt = t[th]
        f["cwsi_last"] = cw[-1] if len(cw) else np.nan
        f["cwsi_mean2"] = cw[-2:].mean() if len(cw) else np.nan
        f["th_age"] = lm + 1 - tt[-1] if len(tt) else 99
        mg = c & (m == 5); mv = v[mg]; mt = t[mg]
        nsel = mv[:, 0] > 0 if len(mv) else np.zeros(0, bool); isel = mv[:, 1] > 0 if len(mv) else np.zeros(0, bool)
        f["n_tot"] = mv[nsel, 0].sum() if len(mv) else 0.0
        f["n_age"] = lm + 1 - mt[nsel][-1] if len(mv) and nsel.any() else 99
        f["irr_tot"] = mv[isel, 1].sum() if len(mv) else 0.0
        f["irr_age"] = lm + 1 - mt[isel][-1] if len(mv) and isel.any() else 99
        tr = c & (m == 6); tv = v[tr, 0]; tt2 = t[tr]
        f["trap_last"] = tv[-1] if len(tv) else np.nan
        f["trap_sum21"] = tv[tt2 >= lm - 20].sum() if len(tv) else np.nan
        rows.append([f[k] for k in FEATS])
    return np.array(rows, np.float32)


def landmark_table(idx, seasons, prep, **kw):
    X, S, LM, D, C = [], [], [], [], []
    for i in idx:
        s = seasons[i]
        if not s["landmarks"]:
            continue
        X.append(landmark_features(s, prep, **kw))
        lms = np.array(s["landmarks"])
        S.append(np.full(len(lms), i)); LM.append(lms)
        if s["T"] is None:
            D.append(np.full(len(lms), 10 ** 4)); C.append(np.full(len(lms), -1))
        else:
            D.append(s["T"] - lms); C.append(np.full(len(lms), s["J"]))
    return dict(X=np.concatenate(X), season=np.concatenate(S), lm=np.concatenate(LM).astype(float),
                delta=np.concatenate(D), cause=np.concatenate(C))


class Imputer:
    def fit(self, X):
        self.mu = np.nanmean(X, 0); self.sd = np.nanstd(X, 0) + 1e-6
        return self

    def __call__(self, X):
        M = np.isnan(X).astype(np.float32)
        Z = (np.where(np.isnan(X), self.mu, X) - self.mu) / self.sd
        return np.hstack([Z, M]).astype(np.float32)


# ------------------------------------------------------------------ rules (agronomic thresholds)
def rule_scores(tab):
    X = tab["X"]; ix = {k: i for i, k in enumerate(FEATS)}
    cw = np.nan_to_num(X[:, ix["cwsi_last"]], nan=0.0)
    ndrop = np.nan_to_num(X[:, ix["ndvi_max"]] - X[:, ix["ndvi_last"]], nan=0.0)
    rat = np.nan_to_num(-X[:, ix["ratio_d"]], nan=0.0)
    pr = np.nan_to_num(X[:, ix["press7"]], nan=0.0)
    soil = np.nan_to_num(X[:, ix["SMFCF"]] - X[:, ix["soil_last"]], nan=0.0) / np.maximum(X[:, ix["SMFCF"]] - X[:, ix["SMW"]], 0.05)
    comp = np.stack([np.maximum((cw - 0.2) / 0.4, (soil - 0.4) / 0.4), np.maximum(rat / 0.08, ndrop / 0.15),
                     pr / 0.5], 1)
    return comp  # per-cause risk scores


# ------------------------------------------------------------------ discrete-time gradient boosting
def hgb_fit(tab, seed=0):
    X, d, c = tab["X"], tab["delta"], tab["cause"]
    rows, ys = [], []
    for i in range(len(X)):
        last = min(d[i], H) if c[i] >= 0 else H
        for h in range(1, last + 1):
            rows.append(np.r_[X[i], h]); ys.append(c[i] + 1 if (c[i] >= 0 and h == d[i]) else 0)
    rows = np.array(rows, np.float32); ys = np.array(ys)
    clf = HistGradientBoostingClassifier(max_iter=250, learning_rate=0.08, max_leaf_nodes=31, random_state=seed,
                                         l2_regularization=1.0)
    clf.fit(rows, ys)
    return clf


def hgb_cif(clf, tab):
    X = tab["X"]; n = len(X)
    probs = np.zeros((n, H, K + 1))
    for h in range(1, H + 1):
        p = clf.predict_proba(np.c_[X, np.full(n, h)])
        full = np.zeros((n, K + 1)); full[:, clf.classes_] = p
        probs[:, h - 1] = full
    surv = np.cumprod(probs[:, :, 0], 1)
    prev = np.concatenate([np.ones((n, 1)), surv[:, :-1]], 1)
    return np.cumsum(prev[:, :, None] * probs[:, :, 1:], 1)


# ------------------------------------------------------------------ cause-specific Cox (lifelines)
def cox_fit(tab, imp):
    from lifelines import CoxPHFitter
    import pandas as pd
    Z = imp(tab["X"])[:, :len(FEATS)]
    d, c = tab["delta"], tab["cause"]
    tt = np.minimum(np.where(c >= 0, d, H), H).astype(float)
    models = []
    for j in range(K):
        e = ((c == j) & (d <= H)).astype(int)
        df = pd.DataFrame(Z, columns=FEATS); df["T"] = tt; df["E"] = e
        df = df.loc[:, df.std() > 1e-6] if True else df
        cph = CoxPHFitter(penalizer=0.1).fit(df, "T", "E")
        models.append(cph)
    return models


def cox_cif(models, tab, imp):
    import pandas as pd
    Z = imp(tab["X"])[:, :len(FEATS)]
    df = pd.DataFrame(Z, columns=FEATS)
    n = len(df); hz = np.zeros((n, H, K))
    for j, m in enumerate(models):
        cols = [c for c in m.params_.index]
        bh = m.baseline_cumulative_hazard_
        grid = np.arange(1, H + 1)
        Lam = np.interp(grid, bh.index.values, bh.values[:, 0])
        dL = np.diff(np.r_[0, Lam])
        risk = np.exp((df[cols].values - m._norm_mean[cols].values) @ m.params_.values)
        hz[:, :, j] = np.clip(risk[:, None] * dL[None, :], 0, 5)
    tot = hz.sum(2)
    surv = np.exp(-np.cumsum(tot, 1))
    prev = np.concatenate([np.ones((n, 1)), surv[:, :-1]], 1)
    frac = np.where(tot[:, :, None] > 0, hz / np.maximum(tot[:, :, None], 1e-12), 0)
    inc = prev[:, :, None] * (1 - np.exp(-tot))[:, :, None] * frac
    return np.cumsum(inc, 1)


# ------------------------------------------------------------------ DeepHit-type MLP and snapshot classifier
class MLPCR(nn.Module):
    def __init__(self, d_in, D=64):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(d_in, D), nn.GELU(), nn.Dropout(0.1), nn.Linear(D, D), nn.GELU())
        self.head = CompetingRiskHead(D, anchor=False)

    def forward(self, x):
        return self.head(self.body(x))


def deephit_rank(logits, delta, cause, sigma=0.5):
    cif = cif_from_logits(logits)  # n H K
    loss = 0.0; cnt = 0
    for j in range(K):
        ev = torch.where((cause == j) & (delta <= H))[0]
        if len(ev) == 0:
            continue
        di = delta[ev]
        Fi = cif[ev, di - 1, j]  # own CIF at own time
        Fall = cif[:, :, j][:, (di - 1)]  # n x m
        comp = (delta[:, None] > di[None, :]).float()
        loss = loss + (comp * torch.exp(-(Fi[None, :] - Fall) / sigma)).sum() / comp.sum().clamp(min=1)
        cnt += 1
    return loss / max(cnt, 1)


def train_mlp(model, Xtr, dtr, ctr, Xva, dva, cva, epochs=60, lr=1e-3, seed=0, rank=0.2, snapshot=False):
    torch.manual_seed(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    Xtr, Xva = torch.tensor(Xtr), torch.tensor(Xva)
    dtr, ctr, dva, cva = map(lambda a: torch.tensor(a, dtype=torch.long), (dtr, ctr, dva, cva))
    rng = np.random.default_rng(seed); best, state, bad = 1e9, None, 0

    def loss_fn(x, d, c):
        out = model(x)
        if snapshot:
            y = torch.where((c >= 0) & (d <= 14), c + 1, torch.zeros_like(c))
            return F.cross_entropy(out, y)
        l = cr_nll(out, d, c)
        return l + rank * deephit_rank(out, d, c) if rank else l
    for ep in range(epochs):
        model.train(); perm = rng.permutation(len(Xtr))
        for i in range(0, len(perm), 256):
            b = perm[i:i + 256]
            l = loss_fn(Xtr[b], dtr[b], ctr[b])
            opt.zero_grad(); l.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            v = loss_fn(Xva, dva, cva).item()
        if v < best - 1e-4:
            best, state, bad = v, {k: x.clone() for k, x in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= 8:
                break
    model.load_state_dict(state)
    return model


class Snapshot(nn.Module):
    """snapshot paradigm of the source study: current-state classifier of stress class within a window"""

    def __init__(self, d_in, D=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, D), nn.GELU(), nn.Dropout(0.1), nn.Linear(D, D), nn.GELU(),
                                 nn.Linear(D, K + 1))

    def forward(self, x):
        return self.net(x)


In [ ]:
%%writefile run_exp.py
"""One experimental run: tier (sim/real) x split x model variant x seed. Saves calibration/test predictions."""
import argparse
import copy
import json
import os
import pickle
import sys
import time
import numpy as np
import torch

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import presage_lib as PL
from presage_lib import PRESAGE, Prep
from train import pretrain, finetune, predict, evaluate_nll, batches
from splits import temporal_split, lozo_split

ROB = {"drop_thermal": dict(drop_mod=4), "drop_optical": dict(drop_mod=3), "drop_soil": dict(drop_mod=2),
       "drop_trap": dict(drop_mod=6), "weather_only": dict(drop_mod=[2, 3, 4, 5, 6]),
       "loss30": dict(token_drop=0.3), "loss50": dict(token_drop=0.5), "loss70": dict(token_drop=0.7)}


def get_data(tier):
    if tier == "sim":
        S = pickle.load(open(os.environ.get("SIMDATA", "data/sim_fieldseasons.pkl"), "rb"))
        if os.environ.get("SMOKE"):
            S = [s for s in S if s["zone"] in (0, 1)][:300] + [s for s in S if s["zone"] in (2, 3)][:300]
    else:
        S = pickle.load(open("data/real_episodes.pkl", "rb"))
        PL.K = 2
        PL.LINK = [(0, 1.0), (None, 0.0)]
        PL.PROXY_MODE = "real"
    return S


def get_split(tier, split, S, seed):
    if tier == "sim":
        if split == "temporal":
            return temporal_split(S)
        return lozo_split(S, int(split[-1]), seed)
    # real: leave-one-consortium-out; remaining consortia split by sensor stream
    c = int(split[-1]); rng = np.random.default_rng(200 + c)
    cons = np.array([s["cons"] for s in S]); code = np.array([s["code"] for s in S])
    test = np.where(cons == c)[0]; rest = np.where(cons != c)[0]
    streams = np.unique(code[rest]); rng.shuffle(streams)
    nv, nc = max(2, int(0.15 * len(streams))), max(3, int(0.25 * len(streams)))
    vs, cs = set(streams[:nv]), set(streams[nv:nv + nc])
    return dict(train=np.array([i for i in rest if code[i] not in vs | cs]),
                val=np.array([i for i in rest if code[i] in vs]),
                cal=np.array([i for i in rest if code[i] in cs]), test=test)


def build_model(variant, D, L=2, lastctx=0):
    kw = dict(D=D, L=L, lastctx=bool(lastctx))
    if variant == "noanchor":
        kw["anchor"] = False
    if variant == "nogap":
        kw["gap_aware"] = False
    if variant == "nosel":
        kw["selective"] = False
    if variant in ("transformer", "grud"):
        kw.update(encoder=variant, anchor=False)
    if variant == "student":
        kw.update(D=24, L=1)
    return PRESAGE(**kw)


def fedavg(model, clients, va, enc, S, rounds_pre=5, rounds_ft=20, seed=0, log=print):
    """FedAvg: each client runs one local epoch per round on its own seasons; weights averaged by size"""
    sizes = np.array([len(c) for c in clients], float); wts = sizes / sizes.sum()
    best, best_state, bad = 1e9, None, 0
    for phase, R in (("pre", rounds_pre), ("ft", rounds_ft)):
        for r in range(R):
            states = []
            for ci, cidx in enumerate(clients):
                local = copy.deepcopy(model)
                if phase == "pre":
                    pretrain(local, cidx, enc, S, epochs=1, seed=seed * 100 + r * 10 + ci, log=lambda *a: None)
                else:
                    local, _ = finetune(local, cidx, cidx[:1] if False else va, enc, S, epochs=1, seed=seed * 100 + r * 10 + ci,
                                        log=lambda *a: None, patience=99)
                states.append(local.state_dict())
            avg = {k: sum(w * s[k].float() for w, s in zip(wts, states)) for k in states[0]}
            model.load_state_dict(avg)
            if phase == "ft":
                v = evaluate_nll(model, va, enc, S)
                log(f"  fed round {r} val {v:.4f}")
                if v < best - 1e-4:
                    best, best_state, bad = v, copy.deepcopy(model.state_dict()), 0
                else:
                    bad += 1
                    if bad >= 5:
                        break
    model.load_state_dict(best_state)
    return model


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--tier", default="sim"); ap.add_argument("--split", default="temporal")
    ap.add_argument("--model", default="presage"); ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--D", type=int, default=64); ap.add_argument("--pre", type=int, default=5)
    ap.add_argument("--ft", type=int, default=30); ap.add_argument("--rob", type=int, default=0)
    ap.add_argument("--threads", type=int, default=1); ap.add_argument("--tag", default="")
    ap.add_argument("--tpre", type=int, default=0)  # 1: self-supervised pretraining also on unlabeled target-domain streams
    ap.add_argument("--L", type=int, default=2)
    ap.add_argument("--lastctx", type=int, default=0)
    a = ap.parse_args()
    torch.set_num_threads(a.threads)
    os.makedirs("results", exist_ok=True); os.makedirs("models", exist_ok=True)
    name = f"{a.tier}_{a.split}_{a.model}_{a.seed}{a.tag}"
    out = f"results/{name}.pkl"
    if os.path.exists(out):
        print("exists", out); return
    S = get_data(a.tier)
    sp = get_split(a.tier, a.split, S, a.seed)
    prep = Prep().fit([S[i] for i in sp["train"]])
    enc = [prep.encode(s) for s in S]
    torch.manual_seed(a.seed); np.random.seed(a.seed)
    t0 = time.time()
    logf = open(f"results/{name}.log", "w")
    log = lambda m: (print(m), logf.write(m + "\n"), logf.flush())
    model = build_model(a.model, a.D, a.L, a.lastctx)
    pre_idx = np.concatenate([sp["train"], sp["val"], sp["cal"], sp["test"]]) if a.tpre else sp["train"]
    if a.model == "fed":
        key = "zone" if a.tier == "sim" else "cons"
        grp = np.array([S[i][key] for i in sp["train"]])
        clients = [sp["train"][grp == g] for g in np.unique(grp)]
        model = fedavg(model, clients, sp["val"], enc, S, seed=a.seed, log=log)
    elif a.model == "local":
        # one model per client (zone); test seasons are scored by the model of their own zone
        key = "zone" if a.tier == "sim" else "cons"
        preds_cal, preds_test = [], []
        res = {}
        for g in np.unique([S[i][key] for i in sp["train"]]):
            m = build_model("presage", a.D, a.L, a.lastctx); torch.manual_seed(a.seed)
            tr = np.array([i for i in sp["train"] if S[i][key] == g])
            va = np.array([i for i in sp["val"] if S[i][key] == g]) if any(S[i][key] == g for i in sp["val"]) else tr[:20]
            pretrain(m, tr, enc, S, epochs=a.pre, seed=a.seed, log=log)
            m, _ = finetune(m, tr, va, enc, S, epochs=a.ft, seed=a.seed, log=log)
            ca = np.array([i for i in sp["cal"] if S[i][key] == g]); te = np.array([i for i in sp["test"] if S[i][key] == g])
            if len(ca): preds_cal.append(predict(m, ca, enc, S))
            if len(te): preds_test.append(predict(m, te, enc, S))
        cat = lambda L: {k: np.concatenate([p[k] for p in L]) for k in L[0]}
        pickle.dump(dict(cal=cat(preds_cal), test=cat(preds_test), time=time.time() - t0, args=vars(a)), open(out, "wb"))
        return
    else:
        if a.model == "student":
            teacher = build_model("presage", a.D, a.L, a.lastctx)
            teacher.load_state_dict(torch.load(f"models/{a.tier}_{a.split}_presage_{a.seed}{a.tag}.pt"))
            teacher.eval()
            model, _ = finetune(model, sp["train"], sp["val"], enc, S, epochs=a.ft, seed=a.seed, teacher=teacher, log=log)
        else:
            if a.model not in ("nopre", "transformer", "grud"):
                pretrain(model, pre_idx, enc, S, epochs=a.pre, seed=a.seed, log=log)
            aux = a.model not in ("transformer", "grud")
            model, _ = finetune(model, sp["train"], sp["val"], enc, S, epochs=a.ft, seed=a.seed, aux=aux, log=log)
    train_time = time.time() - t0
    torch.save(model.state_dict(), f"models/{name}.pt")
    res = dict(cal=predict(model, sp["cal"], enc, S), test=predict(model, sp["test"], enc, S), time=train_time,
               params=sum(p.numel() for p in model.parameters()), args=vars(a))
    # inference latency per landmark (single-thread CPU)
    torch.set_num_threads(1)
    t1 = time.time(); _ = predict(model, sp["test"][:40], enc, S, bs=1)
    nl = sum(len(S[i]["landmarks"]) for i in sp["test"][:40])
    res["latency_ms_per_season"] = 1000 * (time.time() - t1) / 40
    res["n_landmarks_latency"] = nl
    if a.rob:
        for k, kw in ROB.items():
            if a.tier == "real" and k in ("drop_thermal", "drop_trap"):
                continue
            res["rob_" + k] = predict(model, sp["test"], enc, S, **kw)
    pickle.dump(res, open(out, "wb"))
    log(f"done {name} {train_time:.0f}s")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile run_tab.py
"""Tabular, rule and snapshot baselines for one tier x split x seed."""
import argparse
import os
import pickle
import sys
import time
import numpy as np
import torch

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import presage_lib as PL
from presage_lib import Prep, season_arrays, cif_from_logits
import baselines_tab as BT
from run_exp import get_data, get_split, ROB

RFEATS = ["depth", "sand", "silt", "clay", "rzd", "soilflag", "day", "tmax7", "rh7", "rain7", "rain30", "et07", "def7",
          "sm07", "ten_last", "ten_d2", "ten_d5", "ten_max10", "ten_age", "ndvi_last", "ndmi_last", "ndmi_d", "rs_age",
          "irr14", "irr_age"]


def real_features(s, drop_mod=None, token_drop=0.0, rng=None):
    t, m, v = season_arrays(s)
    keep = np.ones(len(t), bool)
    if drop_mod is not None:
        for dm in np.atleast_1d(drop_mod):
            keep &= m != dm
    if token_drop > 0:
        keep &= ~((rng.random(len(t)) < token_drop) & (m != 0))
    t, m, v = t[keep], m[keep], v[keep]
    rows = []
    for lm in s["landmarks"]:
        c = t < lm + 1.0
        f = dict(zip(RFEATS[:6], v[m == 0][0][:6])); f["day"] = lm
        w = c & (m == 1); wl = v[w]; wt = t[w]; r7 = wt >= lm - 6; r30 = wt >= lm - 29
        f["tmax7"] = wl[r7, 1].mean() if r7.any() else np.nan
        f["rh7"] = wl[r7, 2].mean() if r7.any() else np.nan
        f["rain7"] = np.expm1(wl[r7, 3]).sum() if r7.any() else np.nan
        f["rain30"] = np.expm1(wl[r30, 3]).sum() if r30.any() else np.nan
        f["et07"] = wl[r7, 6].mean() if r7.any() else np.nan
        f["def7"] = (wl[r7, 6] - np.expm1(wl[r7, 3])).mean() if r7.any() else np.nan
        f["sm07"] = wl[-1, 7] if len(wl) else np.nan
        so = c & (m == 2); sv = v[so, 0]; st = t[so]
        f["ten_last"] = sv[-1] if len(sv) else np.nan
        f["ten_d2"] = sv[-1] - sv[st <= st[-1] - 2][-1] if len(sv) and (st <= st[-1] - 2).any() else np.nan
        f["ten_d5"] = sv[-1] - sv[st <= st[-1] - 5][-1] if len(sv) and (st <= st[-1] - 5).any() else np.nan
        f["ten_max10"] = sv[st >= lm - 9].max() if len(sv) and (st >= lm - 9).any() else np.nan
        f["ten_age"] = lm + 1 - st[-1] if len(st) else 99
        op = c & (m == 3); ov = v[op]; ot = t[op]
        f["ndvi_last"] = ov[-1, 0] if len(ov) else np.nan
        f["ndmi_last"] = ov[-1, 1] if len(ov) else np.nan
        f["ndmi_d"] = ov[-1, 1] - ov[-2, 1] if len(ov) > 1 else np.nan
        f["rs_age"] = lm + 1 - ot[-1] if len(ot) else 99
        mg = c & (m == 5); mv = v[mg]; mt = t[mg]
        f["irr14"] = mv[mt >= lm - 13, 1].sum() if len(mv) else 0.0
        f["irr_age"] = lm + 1 - mt[-1] if len(mt) else 99
        rows.append([f[k] for k in RFEATS])
    return np.array(rows, np.float32)


def table(tier, idx, S, prep, **kw):
    if tier == "sim":
        if "token_drop" in kw:
            rng = np.random.default_rng(99)
            tabs = []
            X, Sx, LM, Dl, C = [], [], [], [], []
            for i in idx:
                s = S[i]
                if not s["landmarks"]:
                    continue
                t, m, v = season_arrays(s)
                keep = ~((rng.random(len(t)) < kw["token_drop"]) & (m != 0))
                X.append(BT.landmark_features(s, prep, keep_mask=keep))
                Sx.append(np.full(len(s["landmarks"]), i)); LM.append(np.array(s["landmarks"]))
                lms, d, c = PL.landmark_targets(s); Dl.append(d); C.append(c)
            return dict(X=np.concatenate(X), season=np.concatenate(Sx), lm=np.concatenate(LM).astype(float),
                        delta=np.concatenate(Dl), cause=np.concatenate(C))
        return BT.landmark_table(idx, S, prep, **kw)
    rng = np.random.default_rng(99)
    X, Sx, LM, Dl, C = [], [], [], [], []
    for i in idx:
        s = S[i]
        X.append(real_features(s, rng=rng, **kw)); Sx.append(np.full(len(s["landmarks"]), i))
        LM.append(np.array(s["landmarks"])); lms, d, c = PL.landmark_targets(s); Dl.append(d); C.append(c)
    return dict(X=np.concatenate(X), season=np.concatenate(Sx), lm=np.concatenate(LM).astype(float),
                delta=np.concatenate(Dl), cause=np.concatenate(C))


def pack(tab, cif):
    return dict(season=tab["season"], lm=tab["lm"], delta=tab["delta"], cause=tab["cause"], cif=cif)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--tier", default="sim"); ap.add_argument("--split", default="temporal")
    ap.add_argument("--seed", type=int, default=0); ap.add_argument("--rob", type=int, default=0)
    ap.add_argument("--tag", default="")
    a = ap.parse_args()
    torch.set_num_threads(1)
    S = get_data(a.tier)
    BT.K = PL.K
    sp = get_split(a.tier, a.split, S, a.seed)
    prep = Prep().fit([S[i] for i in sp["train"]])
    T = {k: table(a.tier, v, S, prep) for k, v in sp.items()}
    if a.tier == "real":
        BT.FEATS = RFEATS
    imp = BT.Imputer().fit(T["train"]["X"])
    out = {}
    t0 = time.time()
    # discrete-time gradient boosting
    clf = BT.hgb_fit(T["train"], seed=a.seed)
    out["hgb"] = dict(cal=pack(T["cal"], BT.hgb_cif(clf, T["cal"])), test=pack(T["test"], BT.hgb_cif(clf, T["test"])),
                      time=time.time() - t0)
    # DeepHit-type MLP
    t0 = time.time()
    torch.manual_seed(a.seed)
    mlp = BT.MLPCR(imp(T["train"]["X"]).shape[1])
    mlp = BT.train_mlp(mlp, imp(T["train"]["X"]), T["train"]["delta"], T["train"]["cause"], imp(T["val"]["X"]),
                       T["val"]["delta"], T["val"]["cause"], seed=a.seed)
    mlp.eval()
    f = lambda tb: cif_from_logits(mlp(torch.tensor(imp(tb["X"])))).detach().numpy()
    out["deephit"] = dict(cal=pack(T["cal"], f(T["cal"])), test=pack(T["test"], f(T["test"])), time=time.time() - t0)
    # snapshot classifier (current-state stress class within the alert window); expressed as a flat CIF
    t0 = time.time()
    torch.manual_seed(a.seed)
    sn = BT.Snapshot(imp(T["train"]["X"]).shape[1], D=64)
    sn.net[-1] = torch.nn.Linear(64, PL.K + 1)
    sn = BT.train_mlp(sn, imp(T["train"]["X"]), T["train"]["delta"], T["train"]["cause"], imp(T["val"]["X"]),
                      T["val"]["delta"], T["val"]["cause"], seed=a.seed, snapshot=True)
    sn.eval()

    def fsn(tb):
        p = torch.softmax(sn(torch.tensor(imp(tb["X"]))), -1).detach().numpy()[:, 1:]
        ramp = np.clip(np.arange(1, PL.H + 1) / 14.0, 0, 1)  # uniform-in-time spreading inside the window
        return p[:, None, :] * ramp[None, :, None]
    out["snapshot"] = dict(cal=pack(T["cal"], fsn(T["cal"])), test=pack(T["test"], fsn(T["test"])), time=time.time() - t0)
    if a.seed == 0:
        # cause-specific Cox (deterministic)
        t0 = time.time()
        try:
            cox = BT.cox_fit(T["train"], imp)
            out["cox"] = dict(cal=pack(T["cal"], BT.cox_cif(cox, T["cal"], imp)), test=pack(T["test"], BT.cox_cif(cox, T["test"], imp)),
                              time=time.time() - t0)
        except Exception as e:
            print("cox failed", e)
        if a.tier == "sim":
            out["rules"] = dict(cal=dict(**pack(T["cal"], None), comp=BT.rule_scores(T["cal"])),
                                test=dict(**pack(T["test"], None), comp=BT.rule_scores(T["test"])))
    if a.rob:
        for k, kw in ROB.items():
            if a.tier == "real" and k in ("drop_thermal", "drop_trap"):
                continue
            tb = table(a.tier, sp["test"], S, prep, **kw)
            out["hgb"]["rob_" + k] = pack(tb, BT.hgb_cif(clf, tb))
            out["deephit"]["rob_" + k] = pack(tb, f(tb))
    os.makedirs("results", exist_ok=True)
    pickle.dump(out, open(f"results/tab_{a.tier}_{a.split}_{a.seed}{a.tag}.pkl", "wb"))
    print("done", a.tier, a.split, a.seed)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile analyze_lib.py
"""Aggregation of saved predictions into metrics (point, alert/CRC, conformal lead-time bounds)."""
import glob
import os
import pickle
import numpy as np
import eval_lib as E

H = 21


def load_run(tier, split, model, seed, tag=""):
    f = f"results/{tier}_{split}_{model}_{seed}{tag}.pkl"
    return pickle.load(open(f, "rb")) if os.path.exists(f) else None


def load_tab(tier, split, seed):
    f = f"results/tab_{tier}_{split}_{seed}.pkl"
    return pickle.load(open(f, "rb")) if os.path.exists(f) else None


def shift_feats(pred, S, tier):
    cif = pred["cif"]
    tot = cif.sum(2)
    f = [tot[:, 6], tot[:, 13], tot[:, H - 1], pred["lm"] / 100.0]
    f += [cif[:, H - 1, j] for j in range(cif.shape[2])]
    return np.column_stack(f)


def full_eval(res, S, tier, alpha=0.1, alpha_m=0.1, window=None, a0=0.1, weighted=True):
    window = window or (14 if tier == "sim" else 7)
    cal, te = res["cal"], res["test"]
    out = E.point_metrics(te, window=window)
    tg = (0, 1, 2) if tier == "sim" else (0,)
    rc = cal["cif"][:, window - 1, list(tg)].sum(1); rt = te["cif"][:, window - 1, list(tg)].sum(1)
    th = E.crc_threshold(cal, S, rc, alpha_m, window=window, targets=tg)
    am, leads = E.alert_metrics(te, S, rt, th, cause_pred=te["cif"][:, H - 1].argmax(1), window=window, targets=tg)
    out.update({"alert_" + k: v for k, v in am.items()}); out["theta"] = th; out["leads"] = leads
    corr = E.lpb_calibrate(cal, alpha, a0)
    lpb = E.base_lower(te["cif"], a0) - corr
    out.update({"lpb_" + k: v for k, v in E.lpb_eval(lpb, te).items()})
    if weighted:
        fc, ft = shift_feats(cal, S, tier), shift_feats(te, S, tier)
        mu, sd = fc.mean(0), fc.std(0) + 1e-6
        lw, _ = E.lpb_weighted(cal, te, (fc - mu) / sd, (ft - mu) / sd, alpha, a0)
        out.update({"wlpb_" + k: v for k, v in E.lpb_eval(lw, te).items()})
    return out


def rule_eval(res, S, alpha_m=0.1, window=14):
    cal, te = res["cal"], res["test"]
    comp_c, comp_t = cal["comp"], te["comp"]
    rc, rt = comp_c.max(1), comp_t.max(1)
    th = E.crc_threshold(cal, S, rc, alpha_m, window=window)
    am, leads = E.alert_metrics(te, S, rt, th, cause_pred=comp_t.argmax(1), window=window)
    out = {"alert_" + k: v for k, v in am.items()}
    out["cindex_by_cause"] = [E.cindex_cr(comp_t[:, j], te["delta"], te["cause"], j) for j in range(3)]
    out["cindex"] = float(np.nanmean(out["cindex_by_cause"])); out["leads"] = leads
    from sklearn.metrics import f1_score
    ev = (te["cause"] >= 0) & (te["delta"] <= H)
    out["f1"] = float(f1_score(te["cause"][ev], comp_t[ev].argmax(1), average="macro"))
    return out


In [ ]:
%%writefile summarize.py
"""Computes every number reported in the manuscript from saved predictions -> results/summary.json"""
import json
import os
import pickle
import sys
import numpy as np
from scipy import stats
import presage_lib as PL
import eval_lib as E
from analyze_lib import load_run, load_tab, full_eval, rule_eval, shift_feats

H = 21
OUT = "results/summary.json"


def mstd(v):
    v = [x for x in v if x is not None and not (isinstance(x, float) and np.isnan(x))]
    return [float(np.mean(v)), float(np.std(v)), len(v)] if v else [np.nan, np.nan, 0]


KEYS = ["cindex", "ibs", "f1", "ece", "alert_miss", "alert_lead_onset", "alert_lead_visible", "alert_false_alert_frac",
        "alert_fa_per_100lm", "alert_alert_rate", "alert_alert_cause_f1", "lpb_coverage", "lpb_coverage_events",
        "lpb_mean_lpb", "wlpb_coverage", "wlpb_coverage_events", "wlpb_mean_lpb"]


def season_ibs(pred):
    """per-season integrated Brier score (all causes), for paired tests"""
    cif, d, c, s = pred["cif"], pred["delta"], pred["cause"], pred["season"]
    hh = np.arange(1, H + 1)
    y = ((c[:, None, None] == np.arange(cif.shape[2])[None, None, :]) & (d[:, None, None] <= hh[None, :, None])).astype(float)
    keep = ~((c == -2) & (d < H))
    b = ((y - cif) ** 2).mean((1, 2))
    out = {}
    for sid in np.unique(s[keep]):
        out[int(sid)] = float(b[keep & (s == sid)].mean())
    return out


def collect(tier, split, S, models, tabmodels, seeds, tab_seeds, rob=False):
    res = {}
    perseason = {}
    for mname in models:
        rows, ps = [], []
        robs = {}
        for sd in seeds:
            r = load_run(tier, split, mname, sd)
            if r is None:
                continue
            o = full_eval(r, S, tier)
            o["time"] = r.get("time"); o["params"] = r.get("params"); o["latency"] = r.get("latency_ms_per_season")
            rows.append(o); ps.append(season_ibs(r["test"]))
            if rob:
                for k in [x for x in r if x.startswith("rob_")]:
                    pm = E.point_metrics(r[k], window=14 if tier == "sim" else 7)
                    robs.setdefault(k, []).append(pm)
        if rows:
            res[mname] = summarize_rows(rows)
            res[mname]["robust"] = {k: {kk: mstd([x[kk] for x in v]) for kk in ("cindex", "ibs", "f1")} for k, v in robs.items()}
            perseason[mname] = avg_dicts(ps)
    for mname in tabmodels:
        rows, ps, robs = [], [], {}
        for sd in tab_seeds:
            t = load_tab(tier, split, sd)
            if t is None or mname not in t:
                continue
            r = t[mname]
            if mname == "rules":
                o = rule_eval(r, S)
            else:
                o = full_eval(r, S, tier)
                ps.append(season_ibs(r["test"]))
            o["time"] = r.get("time")
            rows.append(o)
            if rob:
                for k in [x for x in r if x.startswith("rob_")]:
                    robs.setdefault(k, []).append(E.point_metrics(r[k], window=14 if tier == "sim" else 7))
        if rows:
            res[mname] = summarize_rows(rows)
            res[mname]["robust"] = {k: {kk: mstd([x[kk] for x in v]) for kk in ("cindex", "ibs", "f1")} for k, v in robs.items()}
            if ps:
                perseason[mname] = avg_dicts(ps)
    return res, perseason


def summarize_rows(rows):
    out = {k: mstd([r.get(k, np.nan) for r in rows]) for k in KEYS + ["time", "params", "latency", "theta"]}
    out["cindex_by_cause"] = [mstd([r["cindex_by_cause"][j] for r in rows]) for j in range(len(rows[0]["cindex_by_cause"]))]
    out["leads"] = [float(x) for x in rows[0].get("leads", [])]
    return out


def avg_dicts(ds):
    keys = set(ds[0]).intersection(*map(set, ds[1:])) if len(ds) > 1 else set(ds[0])
    return {k: float(np.mean([d[k] for d in ds])) for k in keys}


def paired_tests(perseason, ref="presage"):
    out = {}
    if ref not in perseason:
        return out
    pv = []
    for mname, d in perseason.items():
        if mname == ref:
            continue
        ks = sorted(set(d) & set(perseason[ref]))
        a = np.array([perseason[ref][k] for k in ks]); b = np.array([d[k] for k in ks])
        st = stats.wilcoxon(a, b)
        diff = b - a
        rbc = (np.sum(diff > 0) - np.sum(diff < 0)) / max(np.sum(diff != 0), 1)  # matched-pairs rank-biserial (sign)
        out[mname] = dict(n=len(ks), ref_mean=float(a.mean()), other_mean=float(b.mean()), p=float(st.pvalue),
                          median_diff=float(np.median(diff)), effect=float(rbc))
        pv.append((mname, st.pvalue))
    # Holm correction
    pv.sort(key=lambda x: x[1]); m = len(pv); prev = 0
    for i, (k, p) in enumerate(pv):
        adj = min(1.0, max(prev, (m - i) * p)); prev = adj
        out[k]["p_holm"] = float(adj)
    # Friedman across all models with complete seasons
    names = list(perseason)
    if len(names) < 3:
        return out
    ks = sorted(set.intersection(*[set(perseason[n]) for n in names]))
    mat = np.array([[perseason[n][k] for n in names] for k in ks])
    fr = stats.friedmanchisquare(*mat.T)
    ranks = stats.rankdata(mat, axis=1).mean(0)
    out["_friedman"] = dict(chi2=float(fr.statistic), p=float(fr.pvalue), n=len(ks),
                            mean_ranks=dict(zip(names, map(float, ranks))),
                            cd=float(nemenyi_cd(len(names), len(ks))))
    return out


def nemenyi_cd(k, n, alpha=0.05):
    q = {2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850, 7: 2.949, 8: 3.031, 9: 3.102, 10: 3.164, 11: 3.219, 12: 3.268}[k]
    return q * np.sqrt(k * (k + 1) / (6.0 * n))


def bootstrap_dc(S, a, b, n_boot=200, seed=0):
    """paired season bootstrap of the difference in mean cause-specific C-index (b - a) using seed-0 predictions"""
    rng = np.random.default_rng(seed)
    sids = np.unique(a["season"]); diffs = []
    idx_a = {s: np.where(a["season"] == s)[0] for s in sids}; idx_b = {s: np.where(b["season"] == s)[0] for s in sids}
    for _ in range(n_boot):
        pick = rng.choice(sids, len(sids))
        ia = np.concatenate([idx_a[s] for s in pick]); ib = np.concatenate([idx_b[s] for s in pick])
        ca = np.nanmean([E.cindex_cr(a["cif"][ia, H - 1, j], a["delta"][ia], a["cause"][ia], j) for j in range(a["cif"].shape[2])])
        cb = np.nanmean([E.cindex_cr(b["cif"][ib, H - 1, j], b["delta"][ib], b["cause"][ib], j) for j in range(b["cif"].shape[2])])
        diffs.append(cb - ca)
    return [float(np.mean(diffs)), float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5))]


def miss_curve(res, S, tier, alphas=(0.05, 0.1, 0.15, 0.2, 0.3), window=14, comp=False):
    cal, te = res["cal"], res["test"]
    out = []
    for am in alphas:
        if comp:
            rc, rt, cp = cal["comp"].max(1), te["comp"].max(1), te["comp"].argmax(1)
        else:
            rc = cal["cif"][:, window - 1].sum(1); rt = te["cif"][:, window - 1].sum(1); cp = te["cif"][:, H - 1].argmax(1)
        th = E.crc_threshold(cal, S, rc, am, window=window)
        m, leads = E.alert_metrics(te, S, rt, th, cause_pred=cp, window=window)
        out.append(dict(alpha=am, miss=m["miss"], fa=m["false_alert_frac"], lead=m["lead_visible"], rate=m["alert_rate"]))
    return out


def coverage_curve(res, alphas=(0.05, 0.1, 0.2, 0.3)):
    out = []
    for a in alphas:
        c = E.lpb_calibrate(res["cal"], a)
        lpb = E.base_lower(res["test"]["cif"]) - c
        ev = E.lpb_eval(lpb, res["test"])
        out.append(dict(alpha=a, coverage=ev["coverage"], coverage_events=ev["coverage_events"], mean=ev["mean_lpb"]))
    return out


def anchor_corr(res, S):
    te = res["test"]; z = te["z"]
    tr = np.array([[S[s]["truth"]["rftra"][int(l)], S[s]["truth"]["ns"][int(l)],
                    np.log10(S[s]["truth"]["L"][int(l)] + S[s]["truth"]["S"][int(l)] + 1e-5)] for s, l in zip(te["season"], te["lm"])])
    return [float(stats.spearmanr(z[:, c], tr[:, c]).statistic) for c in range(3)], z, tr


def main():
    summ = {}
    if os.path.exists(OUT):
        summ = json.load(open(OUT))
    which = sys.argv[1:] or ["sim", "real", "lozo"]
    if "sim" in which:
        S = pickle.load(open("data/sim_fieldseasons.pkl", "rb"))
        models = ["presage", "student", "transformer", "grud", "nopre", "noanchor", "nogap", "nosel", "fed", "local"]
        res, ps = collect("sim", "temporal", S, models, ["hgb", "deephit", "snapshot", "cox", "rules"], range(5), range(5), rob=True)
        summ["sim_temporal"] = res
        main_set = {k: v for k, v in ps.items() if k in ("presage", "transformer", "grud", "hgb", "deephit", "snapshot", "cox")}
        summ["sim_tests"] = paired_tests(main_set)
        abl_set = {k: v for k, v in ps.items() if k in ("presage", "nopre", "noanchor", "nogap", "nosel")}
        summ["sim_abl_tests"] = paired_tests(abl_set)
        p0 = load_run("sim", "temporal", "presage", 0)
        if p0 is not None:
            t0 = load_tab("sim", "temporal", 0)
            summ["sim_dc"] = {}
            for name, other in [("hgb", t0["hgb"]["test"] if t0 else None), ("transformer", (load_run("sim", "temporal", "transformer", 0) or {}).get("test")),
                                ("grud", (load_run("sim", "temporal", "grud", 0) or {}).get("test"))]:
                if other is not None:
                    summ["sim_dc"][name] = bootstrap_dc(S, p0["test"], other)
            summ["sim_miss_curve"] = {"presage": miss_curve(p0, S, "sim")}
            if t0:
                summ["sim_miss_curve"]["hgb"] = miss_curve(t0["hgb"], S, "sim")
                summ["sim_miss_curve"]["rules"] = miss_curve(t0["rules"], S, "sim", comp=True)
            summ["sim_cov_curve"] = {"presage": coverage_curve(p0)}
            if t0:
                summ["sim_cov_curve"]["hgb"] = coverage_curve(t0["hgb"])
            corr, _, _ = anchor_corr(p0, S)
            summ["anchor_corr"] = corr
    if "sens" in which:
        out = {}
        for tag, fn in [("_k0", "data/sim_fieldseasons_k0.pkl"), ("", "data/sim_fieldseasons.pkl"), ("_k2", "data/sim_fieldseasons_k2.pkl")]:
            if not os.path.exists(fn):
                continue
            Sk = pickle.load(open(fn, "rb"))
            row = {}
            p = load_run("sim", "temporal", "presage", 0, tag)
            t = load_tab("sim", "temporal", 0) if tag == "" else (pickle.load(open(f"results/tab_sim_temporal_0{tag}.pkl", "rb"))
                                                                if os.path.exists(f"results/tab_sim_temporal_0{tag}.pkl") else None)
            for name, res in [("presage", p), ("hgb", t["hgb"] if t else None)]:
                if res is None:
                    continue
                o = full_eval(res, Sk, "sim", weighted=False)
                # pathogen-only lead before visible symptoms
                import eval_lib as E2
                rc = res["cal"]["cif"][:, 13].sum(1); rt = res["test"]["cif"][:, 13].sum(1)
                th = E2.crc_threshold(res["cal"], Sk, rc, 0.1)
                tab = E2.season_alert_table(res["test"], Sk, rt, th)
                lp = [Sk[s_]["vis"][2] - v["first"] for s_, v in tab.items() if v["ev"] and v["hit"] and Sk[s_]["J"] == 2
                      and Sk[s_]["vis"][2] is not None]
                row[name] = dict(cindex=o["cindex"], cP=o["cindex_by_cause"][2], miss=o["alert_miss"],
                                 lead=o["alert_lead_visible"], leadP=float(np.median(lp)) if lp else float("nan"))
            out[{"_k0": "0", "": "1", "_k2": "2"}[tag]] = row
        summ["sens_kappa"] = out
        S = pickle.load(open("data/sim_fieldseasons.pkl", "rb"))
        p0 = load_run("sim", "temporal", "presage", 0); t0 = load_tab("sim", "temporal", 0)
        ws = {}
        for name, res in [("presage", p0), ("hgb", t0["hgb"])]:
            ws[name] = []
            for w in (7, 14, 21):
                o = full_eval(res, S, "sim", window=w, weighted=False)
                ws[name].append(dict(W=w, miss=o["alert_miss"], fa=o["alert_false_alert_frac"], lead=o["alert_lead_visible"],
                                     leadon=o["alert_lead_onset"]))
        summ["sens_window"] = ws
    if "lozo" in which:
        S = pickle.load(open("data/sim_fieldseasons.pkl", "rb"))
        summ["lozo"] = {}
        for z in range(4):
            res, _ = collect("sim", f"lozo{z}", S, ["presage"], ["hgb", "deephit", "cox"], [0], [0])
            summ["lozo"][str(z)] = res
    if "real" in which:
        S = pickle.load(open("data/real_episodes.pkl", "rb"))
        PL.K = 2
        summ["real"] = {}
        for f in range(3):
            res, ps = collect("real", f"loco{f}", S, ["presage", "nopre", "noanchor", "transformer", "grud", "fed", "nogap"],
                              ["hgb", "deephit", "snapshot", "cox"], range(3), range(3), rob=True)
            summ["real"][str(f)] = res
    json.dump(summ, open(OUT, "w"), indent=1, default=float)
    print("saved", OUT)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile figstyle.py
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "serif", "font.serif": ["Liberation Serif", "TeX Gyre Termes", "DejaVu Serif"],
                     "mathtext.fontset": "stix", "font.size": 7.5, "axes.linewidth": 0.6, "xtick.major.width": 0.6,
                     "ytick.major.width": 0.6, "axes.edgecolor": "black", "legend.frameon": False, "savefig.dpi": 300,
                     "axes.spines.top": False, "axes.spines.right": False})


In [ ]:
%%writefile fig_arch.py
"""Fig. 1: PRESAGE architecture (grayscale). Forward data flow = solid arrows; reverse-mode gradient
paths from the training objectives = dashed arrows."""
import figstyle  # noqa
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig = plt.figure(figsize=(7.16, 4.05), dpi=300)
ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, 100); ax.set_ylim(0, 56.5); ax.axis("off")

BOX = {}


def box(key, x, y, w, h, text, fill="white", lw=0.8, style="round,pad=0.02,rounding_size=0.8", fs=7.0, bold=False, ls="-"):
    p = FancyBboxPatch((x, y), w, h, boxstyle=style, fc=fill, ec="black", lw=lw, ls=ls)
    ax.add_patch(p)
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=fs, fontweight="bold" if bold else "normal",
            linespacing=1.15)
    BOX[key] = (x, y, w, h)


def side(key, s, f=0.5):
    x, y, w, h = BOX[key]
    return {"l": (x, y + f * h), "r": (x + w, y + f * h), "t": (x + f * w, y + h), "b": (x + f * w, y)}[s]


def arrow(p0, p1, dashed=False, lw=0.8, rad=0.0, head=6):
    a = FancyArrowPatch(p0, p1, arrowstyle="-|>", mutation_scale=head, lw=lw, color="black",
                        ls=(0, (3.0, 2.0)) if dashed else "-", connectionstyle=f"arc3,rad={rad}", shrinkA=0, shrinkB=0)
    ax.add_patch(a)


def poly(pts, dashed=False, lw=0.8, head=6):
    """orthogonal polyline with an arrow head on the last segment"""
    for (x0, y0), (x1, y1) in zip(pts[:-2], pts[1:-1]):
        ax.plot([x0, x1], [y0, y1], color="black", lw=lw, ls=(0, (3.0, 2.0)) if dashed else "-", solid_capstyle="butt")
    arrow(pts[-2], pts[-1], dashed=dashed, lw=lw, head=head)


# ---------------- input modalities (left column)
mods = ["Weather station (daily)", "Soil-moisture probe (outages)", "Red-edge / NDVI (5-d, clouds)",
        "UAV thermal (irregular)", "Spore trap (weekly)", "Management log (events)", "Static field descriptors"]
ytop = 51.0; hm = 4.6; gap = 1.75
ax.text(10.3, 54.4, "Asynchronous field streams", ha="center", va="center", fontsize=7.4, fontweight="bold")
for i, m in enumerate(mods):
    box(f"m{i}", 1.0, ytop - i * (hm + gap) - hm, 18.6, hm, m, fill="0.93", fs=6.6)
# brace collector line
xc = 21.2
ys = [side(f"m{i}", "r")[1] for i in range(len(mods))]
for i in range(len(mods)):
    ax.plot([side(f"m{i}", "r")[0], xc], [ys[i], ys[i]], color="black", lw=0.7)
ax.plot([xc, xc], [ys[-1], ys[0]], color="black", lw=0.7)

# ---------------- tokenizer and encoder
box("tok", 23.5, 30.0, 14.2, 13.0, "Event tokenizer\n$(t_k,\\,m_k,\\,\\mathbf{x}_k)\\rightarrow\\mathbf{e}_k$\nmodality + time\nembeddings", fs=6.8)
arrow((xc, 36.5), side("tok", "l"))
box("enc", 41.0, 26.0, 16.5, 21.0,
    "Gap-aware selective\nstate-space encoder\n($L$ layers)\n\n$\\bar{\\mathbf{a}}_k=e^{-\\boldsymbol{\\lambda}(\\Delta t_k+\\delta_k)}$\n$\\mathbf{h}_k=\\bar{\\mathbf{a}}_k\\odot\\mathbf{h}_{k-1}+\\bar{\\mathbf{b}}_k\\odot\\mathbf{u}_k$",
    fs=6.8)
arrow(side("tok", "r"), side("enc", "l", 0.4762))
box("lm", 61.0, 33.0, 10.5, 7.0, "Landmark\nstate $\\mathbf{r}(\\tau)$", fs=6.8)
arrow(side("enc", "r", 0.4762), side("lm", "l"))

# ---------------- heads
box("anc", 61.0, 45.0, 17.5, 8.5, "Physiology anchor $\\mathbf{z}$\nCWSI, $\\chi$ (red-edge), $\\pi$", fill="0.93", fs=6.6)
box("cr", 75.5, 31.0, 23.0, 11.0, "Competing-risk hazard head\n$p_h(j\\mid\\mathbf{r},\\mathbf{z})$,  $j\\in\\{\\mathrm{W},\\mathrm{N},\\mathrm{P}\\}$\n$F_j(h)$, $h=1,\\ldots,H$", fs=6.6)
arrow(side("lm", "t"), (side("lm", "t")[0], 45.0))
arrow(side("lm", "r"), side("cr", "l", 0.3636))
# anchor -> hazard (signed link)
poly([side("anc", "r"), (87.0, side("anc", "r")[1]), (87.0, 42.0)])
ax.text(88.0, 47.0, "signed\nlink", fontsize=6.2, ha="left", va="center")

# ---------------- decision layer
box("lpb", 75.5, 17.5, 23.0, 8.0, "Shift-weighted conformal\nlead-time bound $L_\\alpha(\\mathbf{x})$", fs=6.6)
box("crc", 75.5, 5.5, 23.0, 8.0, "Risk-controlled alert\n$R_\\tau\\geq\\hat{\\theta}$, missed onsets $\\leq\\alpha_m$", fs=6.6)
arrow(side("cr", "b"), side("lpb", "t"))
arrow(side("lpb", "b"), side("crc", "t"))

# ---------------- training objectives (bottom) with dashed gradient paths
box("lmf", 23.5, 13.5, 14.2, 8.0, "Masked cross-modal\nforecasting $\\mathcal{L}_{\\mathrm{MF}}$", fill="0.86", fs=6.6)
box("lanc", 41.0, 13.5, 14.2, 8.0, "Predictive anchoring\n$\\mathcal{L}_{\\mathrm{anc}}$", fill="0.86", fs=6.6)
box("lcr", 58.5, 13.5, 12.6, 8.0, "Competing-risk\nlikelihood $\\mathcal{L}_{\\mathrm{CR}}$", fill="0.86", fs=6.6)
box("fed", 23.5, 2.0, 22.0, 7.5, "Federated averaging of all\nparameters $\\Theta$ (data stay on farm)", fs=6.6, ls=(0, (1.5, 1.2)))
box("kd", 49.0, 2.0, 21.0, 7.5, "Distilled edge student\n$\\mathcal{L}_{\\mathrm{KD}}=\\mathrm{KL}(p^{T}\\Vert p^{S})$", fs=6.6, ls=(0, (1.5, 1.2)))
# dashed reverse-mode gradient arrows into the encoder (and the anchor)
arrow(side("lmf", "t"), (side("lmf", "t")[0], 30.0), dashed=True)           # into tokenizer/encoder stack
arrow(side("lanc", "t"), (side("lanc", "t")[0], 26.0), dashed=True)
poly([side("lcr", "t"), (side("lcr", "t")[0], 24.0), (55.0, 24.0), (55.0, 26.0)], dashed=True)
# hazard output feeds the likelihood and, as teacher probabilities, the distilled student (forward, solid)
xv = 74.1
yh = side("cr", "l", 0.1)[1]
ax.plot([side("cr", "l", 0.1)[0], xv], [yh, yh], color="black", lw=0.8)
ax.plot([xv, xv], [yh, side("kd", "r")[1]], color="black", lw=0.8)
arrow((xv, side("lcr", "r", 0.6875)[1]), side("lcr", "r", 0.6875))
arrow((xv, side("kd", "r")[1]), side("kd", "r"))
ax.plot([xv], [side("lcr", "r", 0.6875)[1]], marker="o", ms=1.8, color="black")

# legend (placed in free space below the input column, no overlap)
lx = 1.0
for yy, lab, kind in [(6.9, "forward flow", "-"), (4.6, "gradient path", "--"), (2.3, "optional deployment", "box")]:
    if kind == "box":
        ax.add_patch(FancyBboxPatch((lx, yy - 0.75), 4.5, 1.5, boxstyle="round,pad=0.02,rounding_size=0.4",
                                    fc="white", ec="black", lw=0.8, ls=(0, (1.5, 1.2))))
    else:
        ax.plot([lx, lx + 4.5], [yy, yy], color="black", lw=0.8, ls=(0, (3.0, 2.0)) if kind == "--" else "-")
    ax.text(lx + 5.2, yy, lab, fontsize=6.2, va="center")
fig.savefig("figures/fig1_architecture.png", dpi=300)
print("ok")


In [ ]:
%%writefile fig_case.py
"""Fig. 2 (case study of two test seasons) and Fig. 4 (learned memory and selectivity)."""
import pickle
import sys
import numpy as np
import torch
import figstyle  # noqa
import matplotlib.pyplot as plt
import matplotlib.ticker as mt
import presage_lib as PL
import eval_lib as E
from analyze_lib import load_run
from train import batches
from splits import temporal_split

S = pickle.load(open("data/sim_fieldseasons.pkl", "rb"))
MODN = ["static", "weather", "soil probe", "optical", "UAV thermal", "management", "spore trap"]


def pick(res, cause):
    te = res["test"]
    best, bs = None, -1
    for s in np.unique(te["season"]):
        sv = S[s]
        if sv["J"] != cause or sv["vis"][cause] is None or sv["T"] < 40:
            continue
        sel = te["season"] == s
        r = te["cif"][sel, 13].sum(1)
        lead = sv["vis"][cause] - sv["T"]
        score = lead + 10 * r[-1] - 5 * r[0]
        if score > bs:
            best, bs = s, score
    return best


def fig_case():
    res = load_run("sim", "temporal", "presage", 0)
    cal = res["cal"]; te = res["test"]
    th = E.crc_threshold(cal, S, cal["cif"][:, 13].sum(1), 0.1)
    seasons = [pick(res, 0), pick(res, 2)]
    fig, axs = plt.subplots(3, 2, figsize=(7.0, 4.05), sharex="col",
                            gridspec_kw=dict(height_ratios=[1.15, 1.0, 1.1], hspace=0.16, wspace=0.08))
    for col, s in enumerate(seasons):
        sv = S[s]; T = sv["T"]; J = sv["J"]; V = sv["vis"][J]
        ax = axs[0, col]
        for m in range(1, 7):
            t = sv["tok_t"][sv["tok_m"] == m]
            ax.plot(t, np.full(len(t), 7 - m), "|", color="black", ms=4.5 if m != 1 else 3.0, mew=0.7)
        ax.set_yticks(range(1, 7)); ax.set_yticklabels(MODN[1:][::-1] if col == 0 else [], fontsize=6)
        ax.set_ylim(0.3, 7.5)
        ax.set_title(["Season A: water stress first", "Season B: pathogen infection first"][col], fontsize=7.4, pad=3)
        ax = axs[1, col]
        d = np.arange(sv["n"]); emerged = sv["truth"]["lai"] >= 0.05
        mask = lambda a: np.where(emerged, a, np.nan)
        ax.plot(d, mask(sv["truth"]["rftra"]), color="black", lw=0.9, label="transpiration ratio")
        ax.plot(d, mask(np.clip(sv["truth"]["ns"], 0, 1.2)), color="0.45", lw=0.9, ls=(0, (4, 1.5)), label="N sufficiency")
        inf = np.clip((sv["truth"]["L"] + sv["truth"]["S"]) / 0.02, 0, 1.2)
        ax.plot(d, mask(inf), color="0.6", lw=0.9, ls=(0, (1, 1)), label="infected area (scaled)")
        ax.set_ylim(-0.08, 1.3); ax.set_yticks([0, 0.5, 1.0]); ax.set_ylabel("simulated state" if col == 0 else "", fontsize=6.6)

        ax = axs[2, col]
        sel = te["season"] == s
        lm = te["lm"][sel]; cif = te["cif"][sel]
        styles = [("water", "black", "-"), ("nitrogen", "0.45", (0, (4, 1.5))), ("pathogen", "0.6", (0, (1, 1)))]
        for j, (nm, c, ls) in enumerate(styles):
            ax.plot(lm, cif[:, 13, j], color=c, ls=ls, lw=1.0, marker="o", ms=2.0, label=f"$F_{{{nm[0].upper()}}}(14)$")
        r = cif[:, 13].sum(1)
        ax.axhline(th, color="black", lw=0.6, ls=(0, (6, 2)))
        al = lm[r >= th]
        if len(al):
            ax.plot(al, np.full(len(al), 1.03), "v", color="black", ms=3.2, clip_on=False)
        ax.set_ylim(0, 1.08); ax.set_ylabel("cumulative incidence" if col == 0 else "", fontsize=6.6)
        ax.set_xlabel("days after sowing", fontsize=6.8)
        if col == 0:
            ax.text(lm[0] - 4, th + 0.03, r"$\hat{\theta}$", fontsize=6.5)
        for a in axs[:, col]:
            a.axvline(T, color="black", lw=0.8)
            if V is not None:
                a.axvline(V, color="0.5", lw=0.8, ls=(0, (2, 1.5)))
            a.tick_params(labelsize=6)
        axs[0, col].text(T - 0.8, 6.95, "onset", fontsize=5.8, ha="right", va="center")
        if V is not None:
            axs[0, col].text(V + 0.8, 6.95, "visible", fontsize=5.8, ha="left", va="center")
        axs[2, col].set_xlim(0, max(V or T, T) + 12)
    h1, l1 = axs[1, 0].get_legend_handles_labels(); h2, l2 = axs[2, 0].get_legend_handles_labels()
    from matplotlib.lines import Line2D
    extra = [Line2D([], [], color="black", lw=0.8), Line2D([], [], color="0.5", lw=0.8, ls=(0, (2, 1.5))),
             Line2D([], [], color="black", ls="none", marker="v", ms=3.2)]
    fig.legend(h1 + h2 + extra, l1 + l2 + ["onset", "visible symptoms", "alert"], loc="lower center", ncol=9,
               fontsize=5.8, handlelength=1.8, columnspacing=1.0, frameon=False, bbox_to_anchor=(0.54, 0.0))
    fig.subplots_adjust(left=0.1, right=0.99, bottom=0.135, top=0.93)
    fig.savefig("figures/fig2_case.png", dpi=300)
    plt.close(fig)
    return seasons, th


def fig_mech():
    sp = temporal_split(S)
    prep = PL.Prep().fit([S[i] for i in sp["train"]])
    enc = [prep.encode(s) for s in S]
    m = PL.PRESAGE(D=64)
    m.load_state_dict(torch.load("models/sim_temporal_presage_0.pt")); m.eval()
    taus = [1.0 / torch.exp(l.log_lam).detach().numpy().ravel() for l in m.enc.layers]
    tau0 = np.exp(np.linspace(np.log(0.7), np.log(60.0), 8))
    caps = {}

    def hook(layer, idx):
        def f(mod, inp, out):
            caps.setdefault(idx, []).append(out.detach())
        return f
    hs = [l.dlt.register_forward_hook(hook(l, i)) for i, l in enumerate(m.enc.layers)]
    deltas = {k: [] for k in range(1, 7)}
    with torch.no_grad():
        for bidx, b in batches(sp["test"][:120], enc, S, 32, np.random.default_rng(0), shuffle=False):
            caps.clear(); m.enc(b)
            dl = torch.nn.functional.softplus(caps[0][0]).mean(-1)  # mean step over channels, layer 1
            for k in range(1, 7):
                sel = (b["m"] == k) & ~b["pad"]
                deltas[k] += dl[sel].tolist()
    for h in hs:
        h.remove()
    fig, axs = plt.subplots(1, 2, figsize=(3.45, 1.85), gridspec_kw=dict(wspace=0.45, width_ratios=[1, 1.1]))
    ax = axs[0]
    init = np.tile(tau0, taus[0].size // tau0.size)
    for li, (tt, col, mk) in enumerate([(taus[0], "black", "o"), (taus[1], "0.55", "s")]):
        ratio = tt / init
        ax.scatter(init * np.exp(np.random.default_rng(li).normal(0, 0.04, init.size)), ratio, s=2, color=col,
                   marker=mk, lw=0, alpha=0.6, label=f"layer {li + 1}")
    ax.axhline(1.0, color="0.4", lw=0.6, ls=(0, (3, 2)))
    ax.set_xscale("log")
    ax.set_xlabel(r"initial time constant (days)", fontsize=6.4); ax.set_ylabel(r"learned / initial $1/\lambda$", fontsize=6.4)
    ax.legend(fontsize=5.5, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, handlelength=1.0, markerscale=2.5,
              columnspacing=0.8, borderaxespad=0.1)
    ax.text(-0.42, 1.02, "(a)", transform=ax.transAxes, fontsize=7.5)
    ax.tick_params(labelsize=6)
    ax = axs[1]
    data = [np.array(deltas[k]) for k in range(1, 7)]
    bp = ax.boxplot(data, showfliers=False, patch_artist=True, widths=0.6, medianprops=dict(color="black", lw=0.9),
                    whiskerprops=dict(lw=0.6), capprops=dict(lw=0.6))
    for b_ in bp["boxes"]:
        b_.set_facecolor("0.8"); b_.set_edgecolor("black"); b_.set_linewidth(0.6)
    ax.set_xticks(range(1, 7)); ax.set_xticklabels(["weather", "soil", "optical", "thermal", "mgmt", "trap"], rotation=35,
                                                   ha="right", fontsize=6)
    ax.set_ylabel(r"mean step $\Delta_k$ (days)", fontsize=6.4)
    ax.text(-0.3, 1.02, "(b)", transform=ax.transAxes, fontsize=7.5)
    fig.subplots_adjust(left=0.15, right=0.99, bottom=0.26, top=0.86)
    fig.savefig("figures/fig4_mechanism.png", dpi=300)
    plt.close(fig)
    return {k: float(np.median(v)) for k, v in deltas.items()}, [float(np.median(t)) for t in taus], \
        [float(np.mean(t > 60)) for t in taus]


if __name__ == "__main__":
    import json
    out = {}
    if "case" in sys.argv:
        s, th = fig_case(); out["case"] = dict(seasons=[int(x) for x in s], theta=th)
    if "mech" in sys.argv:
        d, t, f = fig_mech(); out["mech"] = dict(delta_median=d, tau_median=t, frac_tau_gt60=f)
    json.dump(out, open("results/fig_case_mech.json", "w"), indent=1)
    print(out)


In [ ]:
%%writefile fig_results.py
"""Result figures (grayscale, no overlapping elements). Reads results/summary.json and saved predictions."""
import json
import pickle
import sys
import numpy as np
import figstyle  # noqa
import matplotlib.pyplot as plt
import matplotlib.ticker as mt
import presage_lib as PL
from analyze_lib import load_run, load_tab

S = json.load(open("results/summary.json"))
NAMES = {"presage": "PRESAGE", "hgb": "Gradient boosting", "deephit": "DeepHit-type MLP", "cox": "Cause-specific Cox",
         "snapshot": "Snapshot classifier", "transformer": "Time-aware Transformer", "grud": "GRU-D", "rules": "Index thresholds",
         "student": "PRESAGE student", "nopre": "w/o pretraining", "noanchor": "w/o anchoring", "nogap": "w/o elapsed time",
         "nosel": "w/o selectivity", "fed": "Federated", "local": "Local only"}
GRAYS = ["0.0", "0.35", "0.55", "0.7", "0.82"]
MARK = ["o", "s", "^", "D", "v", "P", "X"]


def save(fig, name):
    fig.savefig(f"figures/{name}.png", dpi=300)
    plt.close(fig)


def fig_forest():
    st = S["sim_temporal"]
    order = [m for m in ["presage", "hgb", "transformer", "grud", "deephit", "cox", "snapshot"] if m in st]
    fig, axs = plt.subplots(1, 3, figsize=(3.45, 2.55), sharey=True, gridspec_kw=dict(wspace=0.12))
    y = np.arange(len(order))[::-1]
    for ax, key, lab in zip(axs, ["cindex", "ibs", "f1"], ["C-index (↑)", "IBS (↓)", "Cause macro-F1 (↑)"]):
        for yi, mname in zip(y, order):
            v = st[mname].get(key, [np.nan, 0])
            if v[0] is None or np.isnan(v[0]):
                continue
            ax.errorbar(v[0], yi, xerr=v[1], fmt="o" if mname != "presage" else "s", color="black",
                        mfc="black" if mname == "presage" else "white", ms=3.6, lw=0.8, capsize=1.8)
        ax.set_xlabel(lab, fontsize=7)
        ax.xaxis.set_major_locator(mt.MaxNLocator(3))
        ax.tick_params(labelsize=6.5)
        ax.grid(axis="x", color="0.88", lw=0.5)
    axs[0].set_yticks(y); axs[0].set_yticklabels([NAMES[m] for m in order], fontsize=6.6)
    fig.subplots_adjust(left=0.34, right=0.98, bottom=0.17, top=0.97)
    save(fig, "fig3_main")


def fig_alerts():
    mc = S.get("sim_miss_curve", {})
    fig, axs = plt.subplots(1, 2, figsize=(3.45, 1.85), gridspec_kw=dict(wspace=0.45))
    ax = axs[0]
    for i, (mname, curve) in enumerate(mc.items()):
        a = [c["alpha"] for c in curve]; m = [c["miss"] for c in curve]
        ax.plot(a, m, marker=MARK[i], color=GRAYS[i], mfc="white" if i else "black", ms=3.2, lw=0.9, label=NAMES[mname])
    ax.plot([0, 0.32], [0, 0.32], ls=(0, (3, 2)), color="0.5", lw=0.7, label="nominal")
    ax.set_xlabel(r"target miss rate $\alpha_m$"); ax.set_ylabel("empirical miss rate")
    ax.set_xlim(0, 0.32); ax.set_ylim(0, 0.42)
    ax.legend(fontsize=5.6, loc="upper left", handlelength=1.6, borderaxespad=0.2)
    ax.text(-0.33, 1.02, "(a)", transform=ax.transAxes, fontsize=7.5)
    ax = axs[1]
    for i, (mname, curve) in enumerate(mc.items()):
        a = [c["alpha"] for c in curve]; f = [c["fa"] for c in curve]
        ax.plot(a, f, marker=MARK[i], color=GRAYS[i], mfc="white" if i else "black", ms=3.2, lw=0.9)
    ax.set_xlabel(r"target miss rate $\alpha_m$"); ax.set_ylabel("false-alert fraction")
    ax.set_xlim(0, 0.32); ax.set_ylim(0, 1.0)
    ax.text(-0.33, 1.02, "(b)", transform=ax.transAxes, fontsize=7.5)
    fig.subplots_adjust(left=0.13, right=0.98, bottom=0.21, top=0.92)
    save(fig, "fig5_alerts")


def fig_leads(sims):
    """lead time before visible symptoms by cause at alpha_m = 0.1 (seed 0)"""
    from analyze_lib import full_eval
    import eval_lib as E
    fig, ax = plt.subplots(figsize=(3.45, 1.75))
    pos, data, labels = [], [], []
    p0 = load_run("sim", "temporal", "presage", 0); t0 = load_tab("sim", "temporal", 0)
    for gi, (mname, res, comp) in enumerate([("presage", p0, False), ("hgb", t0["hgb"], False), ("rules", t0["rules"], True)]):
        cal, te = res["cal"], res["test"]
        if comp:
            rc, rt = cal["comp"].max(1), te["comp"].max(1)
        else:
            rc, rt = cal["cif"][:, 13].sum(1), te["cif"][:, 13].sum(1)
        th = E.crc_threshold(cal, sims, rc, 0.1)
        tab = E.season_alert_table(te, sims, rt, th)
        for j, cn in enumerate(["water", "nitrogen", "pathogen"]):
            leads = [sims[s]["vis"][j] - v["first"] for s, v in tab.items() if v["ev"] and v["hit"] and sims[s]["J"] == j
                     and sims[s]["vis"][j] is not None]
            data.append(leads); pos.append(j * 4 + gi)
    bp = ax.boxplot(data, positions=pos, widths=0.7, patch_artist=True, showfliers=False,
                    medianprops=dict(color="black", lw=1.0), whiskerprops=dict(lw=0.7), capprops=dict(lw=0.7))
    for i, b in enumerate(bp["boxes"]):
        b.set_facecolor(["0.25", "0.65", "white"][i // 3]); b.set_edgecolor("black"); b.set_linewidth(0.7)
    ax.set_xticks([1, 5, 9]); ax.set_xticklabels(["Water", "Nitrogen", "Pathogen"])
    ax.set_ylabel("days before visible symptoms")
    ax.axhline(0, color="0.6", lw=0.6, ls=(0, (2, 2)))
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(fc=c, ec="black", lw=0.6, label=l) for c, l in
                       zip(["0.25", "0.65", "white"], ["PRESAGE", "Gradient boosting", "Index thresholds"])],
              fontsize=5.8, loc="upper center", bbox_to_anchor=(0.5, 1.2), ncol=3, columnspacing=1.0, handlelength=1.2)
    fig.subplots_adjust(left=0.13, right=0.98, bottom=0.14, top=0.84)
    save(fig, "fig6_leads")


def fig_robust():
    st = S["sim_temporal"]
    conds = ["clean", "rob_drop_thermal", "rob_drop_optical", "rob_drop_soil", "rob_drop_trap", "rob_weather_only"]
    lab = ["none", "thermal", "optical", "soil probe", "spore trap", "all but\nweather"]
    conds2 = ["clean", "rob_loss30", "rob_loss50", "rob_loss70"]
    fig, axs = plt.subplots(1, 2, figsize=(3.45, 2.05), gridspec_kw=dict(width_ratios=[1.6, 1], wspace=0.32))
    models = [m for m in ["presage", "hgb", "transformer", "grud", "deephit"] if m in st]
    for i, mname in enumerate(models):
        r = st[mname]
        vals = [r["cindex"][0]] + [r["robust"].get(c, {}).get("cindex", [np.nan])[0] for c in conds[1:]]
        axs[0].plot(range(len(conds)), vals, marker=MARK[i], color=GRAYS[i], mfc="white" if i else "black", ms=3.0, lw=0.8,
                    label=NAMES[mname])
        vals2 = [r["cindex"][0]] + [r["robust"].get(c, {}).get("cindex", [np.nan])[0] for c in conds2[1:]]
        axs[1].plot([0, 30, 50, 70], vals2, marker=MARK[i], color=GRAYS[i], mfc="white" if i else "black", ms=3.0, lw=0.8)
    axs[0].set_xticks(range(len(conds))); axs[0].set_xticklabels(lab, rotation=35, ha="right", fontsize=6)
    axs[0].set_ylabel("C-index"); axs[0].set_xlabel("removed modality", fontsize=6.6)
    axs[1].set_xlabel("random token loss (%)", fontsize=6.6); axs[1].set_xticks([0, 30, 50, 70])
    lo = min(min(l.get_ydata().min() for l in a.get_lines()) for a in axs)
    for a in axs:
        a.set_ylim(np.floor(lo * 20) / 20 - 0.02, 0.9)
        a.grid(axis="y", color="0.9", lw=0.5)
    axs[0].legend(fontsize=5.5, loc="upper center", bbox_to_anchor=(0.8, 1.27), ncol=3, columnspacing=0.8, handlelength=1.5)
    fig.subplots_adjust(left=0.12, right=0.98, bottom=0.3, top=0.82)
    save(fig, "fig7_robust")


def fig_anchor(sims):
    from summarize import anchor_corr
    p0 = load_run("sim", "temporal", "presage", 0)
    corr, z, tr = anchor_corr(p0, sims)
    fig, axs = plt.subplots(1, 3, figsize=(3.45, 1.45), gridspec_kw=dict(wspace=0.55))
    labs = [("$z_1$ (CWSI channel)", "transpiration ratio"), ("$z_2$ (chlorophyll channel)", "N sufficiency"),
            ("$z_3$ (infection channel)", "log$_{10}$ infected area")]
    for c, ax in enumerate(axs):
        ax.hexbin(z[:, c], tr[:, c], gridsize=22, cmap="Greys", mincnt=1, bins="log", linewidths=0.1)
        ax.set_xlabel(labs[c][0], fontsize=5.8); ax.set_ylabel(labs[c][1], fontsize=5.8)
        ax.tick_params(labelsize=5.5)
        ax.set_title(r"$\rho_s$ = %.2f" % corr[c], fontsize=6.2, pad=2)
        ax.xaxis.set_major_locator(mt.MaxNLocator(3)); ax.yaxis.set_major_locator(mt.MaxNLocator(3))
    fig.subplots_adjust(left=0.1, right=0.99, bottom=0.27, top=0.86)
    save(fig, "fig8_anchor")


def fig_coverage():
    lz = S.get("lozo", {})
    fig, axs = plt.subplots(1, 2, figsize=(3.45, 1.85), gridspec_kw=dict(wspace=0.42))
    ax = axs[0]
    zn = ["coastal", "plain", "foothill", "alpine"]
    xs = np.arange(4); w = 0.36
    sc = [lz.get(str(z), {}).get("presage", {}).get("lpb_coverage_events", [np.nan])[0] for z in range(4)]
    wc = [lz.get(str(z), {}).get("presage", {}).get("wlpb_coverage_events", [np.nan])[0] for z in range(4)]
    ax.bar(xs - w / 2, sc, w, color="0.75", ec="black", lw=0.5, label="split conformal")
    ax.bar(xs + w / 2, wc, w, color="0.25", ec="black", lw=0.5, label="shift-weighted")
    ax.axhline(0.9, color="black", lw=0.7, ls=(0, (3, 2)))
    ax.set_xticks(xs); ax.set_xticklabels(zn, fontsize=6, rotation=25, ha="right")
    ax.set_ylabel("coverage of onsets"); ax.set_ylim(0.6, 1.02)
    ax.legend(fontsize=5.5, loc="upper center", bbox_to_anchor=(0.5, 1.28), ncol=2, columnspacing=0.8, handlelength=1.2)
    ax.text(-0.36, 1.06, "(a)", transform=ax.transAxes, fontsize=7.5)
    ax = axs[1]
    cc = S.get("sim_cov_curve", {})
    for i, (mname, curve) in enumerate(cc.items()):
        a = [1 - c["alpha"] for c in curve]
        ax.plot(a, [c["coverage_events"] for c in curve], marker=MARK[i], color=GRAYS[i], mfc="white" if i else "black",
                ms=3.2, lw=0.8, label=NAMES[mname])
    ax.plot([0.68, 0.97], [0.68, 0.97], color="0.5", lw=0.7, ls=(0, (3, 2)))
    ax.set_xlabel(r"nominal $1-\alpha$"); ax.set_ylabel("empirical coverage")
    ax.set_xlim(0.67, 0.97); ax.set_ylim(0.6, 1.0)
    ax.legend(fontsize=5.5, loc="upper left", handlelength=1.4, borderaxespad=0.2)
    ax.text(-0.36, 1.02, "(b)", transform=ax.transAxes, fontsize=7.5)
    fig.subplots_adjust(left=0.14, right=0.98, bottom=0.25, top=0.83)
    save(fig, "fig9_coverage")


if __name__ == "__main__":
    sims = pickle.load(open("data/sim_fieldseasons.pkl", "rb"))
    for f in sys.argv[1:]:
        if f in ("leads", "anchor"):
            globals()["fig_" + f](sims)
        else:
            globals()["fig_" + f]()
    print("ok")


In [ ]:
%%writefile fig_extra.py
"""Fig. 10 (real consortia) and Fig. 11 (accuracy versus model size)."""
import json
import sys
import numpy as np
import figstyle  # noqa
import matplotlib.pyplot as plt

S = json.load(open("results/summary.json"))
NAMES = {"presage": "PRESAGE", "hgb": "Gradient boosting", "deephit": "DeepHit-type MLP", "cox": "Cox",
         "transformer": "Transformer", "grud": "GRU-D", "nopre": "PRESAGE w/o pretraining", "student": "Student",
         "snapshot": "Snapshot"}
FILLS = ["0.0", "0.3", "0.5", "0.7", "0.85", "white"]
HATCH = ["", "", "", "", "", "////"]


def fig_real():
    rr = S["real"]
    models = [k for k in ["presage", "grud", "transformer", "hgb", "deephit", "cox"] if any(k in rr.get(str(f), {}) for f in range(3))]
    fig, axs = plt.subplots(1, 2, figsize=(3.45, 1.95), gridspec_kw=dict(width_ratios=[1.6, 0.8], wspace=0.75))
    ax = axs[0]
    w = 0.8 / len(models); xs = np.arange(3)
    for i, k in enumerate(models):
        mu = [rr.get(str(f), {}).get(k, {}).get("cindex", [np.nan, 0])[0] for f in range(3)]
        sd = [rr.get(str(f), {}).get(k, {}).get("cindex", [np.nan, 0])[1] for f in range(3)]
        ax.bar(xs - 0.4 + w / 2 + i * w, mu, w, yerr=sd, color=FILLS[i], ec="black", lw=0.4, hatch=HATCH[i],
               error_kw=dict(lw=0.5, capsize=1.2), label=NAMES[k])
    ax.axhline(0.5, color="0.5", lw=0.6, ls=(0, (3, 2)))
    ax.set_xticks(xs); ax.set_xticklabels(["C0 held out", "C1 held out", "C2 held out"], fontsize=6.2)
    ax.set_ylabel("C-index"); ax.set_ylim(0.4, 0.9)
    ax.legend(fontsize=5.4, ncol=3, loc="lower center", bbox_to_anchor=(0.75, 1.0), columnspacing=0.7, handlelength=1.1)
    ax.text(-0.30, 1.04, "(a)", transform=ax.transAxes, fontsize=7.5)
    ax = axs[1]
    for i, k in enumerate(models):
        mi = [rr.get(str(f), {}).get(k, {}).get("alert_miss", [np.nan])[0] for f in range(3)]
        ax.barh(len(models) - 1 - i, np.nanmean(mi), 0.7, color=FILLS[i], ec="black", lw=0.4, hatch=HATCH[i])
    ax.axvline(0.1, color="0.4", lw=0.7, ls=(0, (3, 2)))
    ax.set_yticks(range(len(models))); ax.set_yticklabels([NAMES[k] for k in models][::-1], fontsize=5.6)
    ax.set_xlabel("miss rate ($\\alpha_m$ = 0.1)", fontsize=6.4); ax.set_xlim(0, 0.55)
    ax.text(-0.75, 1.02, "(b)", transform=ax.transAxes, fontsize=7.5)
    fig.subplots_adjust(left=0.12, right=0.97, bottom=0.17, top=0.8)
    fig.savefig("figures/fig10_real.png", dpi=300); plt.close(fig)


def fig_eff():
    st = S["sim_temporal"]
    fig, ax = plt.subplots(figsize=(3.45, 1.9))
    pts = [k for k in ["presage", "student", "transformer", "grud"] if k in st]
    offs = {"presage": (8, 0), "student": (-7, 0), "transformer": (9, -6), "grud": (8, 6)}
    has = {"presage": "left", "student": "right", "transformer": "left", "grud": "left"}
    for i, k in enumerate(pts):
        x = st[k]["latency"][0]; y = st[k]["cindex"][0]
        ax.scatter(x, y, s=12 + st[k]["params"][0] / 1500, color=FILLS[i], ec="black", lw=0.6, zorder=3)
        ax.annotate(f"{NAMES[k]} ({st[k]['params'][0]/1e3:.0f}k)", (x, y), xytext=offs[k], textcoords="offset points",
                    fontsize=5.8, va="center", ha=has[k])
    ax.set_xlabel("CPU latency per season (ms, single thread)"); ax.set_ylabel("C-index")
    lo = min(st[k]["cindex"][0] for k in pts); hi = max(st[k]["cindex"][0] for k in pts)
    ax.set_ylim(lo - 0.008, hi + 0.008); ax.set_xlim(0, 14)
    fig.subplots_adjust(left=0.14, right=0.97, bottom=0.21, top=0.95)
    fig.savefig("figures/fig11_eff.png", dpi=300); plt.close(fig)


if __name__ == "__main__":
    for a in sys.argv[1:]:
        globals()["fig_" + a]()
    print("ok")


In [ ]:
%%writefile rob_ablation.py
"""Robustness of ablated variants (inference only, saved weights): does elapsed-time discretization or anchoring
help when modalities disappear?"""
import json, pickle, sys
import numpy as np, torch
import presage_lib as PL
from run_exp import build_model, ROB
from train import predict
from splits import temporal_split
import eval_lib as E
torch.set_num_threads(1)
S = pickle.load(open("data/sim_fieldseasons.pkl", "rb"))
sp = temporal_split(S)
prep = PL.Prep().fit([S[i] for i in sp["train"]]); enc = [prep.encode(s) for s in S]
conds = ["weather_only", "drop_soil", "drop_thermal", "loss50", "loss70"]
out = {}
for v in ["presage", "nogap", "noanchor", "nopre", "nosel"]:
    for seed in [0, 1]:
        m = build_model(v, 64); m.load_state_dict(torch.load(f"models/sim_temporal_{v}_{seed}.pt")); m.eval()
        for c in conds:
            p = predict(m, sp["test"], enc, S, **ROB[c])
            out.setdefault(v, {}).setdefault(c, []).append(E.point_metrics(p)["cindex"])
    print(v, {c: round(float(np.mean(x)), 4) for c, x in out[v].items()}, flush=True)
json.dump({v: {c: [float(np.mean(x)), float(np.std(x))] for c, x in d.items()} for v, d in out.items()},
          open("results/rob_ablation.json", "w"), indent=1)


## 4. Build the two data tiers
Tier 1 drives WOFOST 8.1 and the epidemic model with real station weather and simulates asynchronous sensors; Tier 2 extracts dry-down episodes from the real tensiometer records.

In [ ]:
!python build_sim.py 2>&1 | grep -v -i logging | tail -15
!python -I build_real.py | tail -10

## 5. Experiments
Runs already present in `results/` are skipped.

In [ ]:
import subprocess
def run(cmd):
    print('>>', cmd); subprocess.run(cmd, shell=True)
R = 'python run_exp.py --threads 2'; T = 'python run_tab.py'; RR = '--pre 8 --ft 40 --D 32 --tpre 1'
NS = [0] if MODE == 'quick' else [0, 1, 2]          # neural seeds
TS = [0] if MODE == 'quick' else [0, 1, 2, 3, 4]    # tabular seeds
# E1, E4, E7: main comparison, robustness, distilled student
for s in TS:
    if not os.path.exists(f'results/tab_sim_temporal_{s}.pkl'): run(f'{T} --tier sim --split temporal --seed {s} --rob 1')
for s in NS:
    run(f'{R} --model presage --seed {s} --rob 1'); run(f'{R} --model student --seed {s}')
    run(f'{R} --model transformer --seed {s} --rob 1'); run(f'{R} --model grud --seed {s} --rob 1')
# E2: ablations (two seeds in the paper)
for s in NS[:2]:
    for m in ['nopre', 'noanchor', 'nogap', 'nosel']: run(f'{R} --model {m} --seed {s}')
# E6: real consortia
for f in range(3):
    for s in NS:
        if not os.path.exists(f'results/tab_real_loco{f}_{s}.pkl'): run(f'{T} --tier real --split loco{f} --seed {s} --rob 1')
        run(f'{R} --tier real --split loco{f} --model presage --seed {s} {RR} --rob 1')
        for m in ['nopre', 'noanchor', 'transformer', 'grud']: run(f'{R} --tier real --split loco{f} --model {m} --seed {s} {RR}')
if MODE == 'full':
    # E5: leave-one-zone-out
    for z in range(4):
        if not os.path.exists(f'results/tab_sim_lozo{z}_0.pkl'): run(f'{T} --tier sim --split lozo{z} --seed 0')
        run(f'{R} --split lozo{z} --model presage --seed 0')
    # E8: strength of the latent-infection signature
    for k in ['0.0', '2.0']:
        t = '_k' + k[0]
        if not os.path.exists(f'data/sim_fieldseasons{t}.pkl'): run(f'KAPPA_BIO={k} TAG={t} python build_sim.py > /dev/null 2>&1')
        run(f'SIMDATA=data/sim_fieldseasons{t}.pkl {T} --tier sim --split temporal --seed 0 --tag {t}')
        run(f'SIMDATA=data/sim_fieldseasons{t}.pkl {R} --model presage --seed 0 --tag {t}')
    # E7: federated and local training
    run(f'{R} --model fed --seed 0'); run(f'{R} --model local --seed 0')
    for f in range(3): run(f'{R} --tier real --split loco{f} --model fed --seed 0 {RR}')
    run('python rob_ablation.py')

## 6. Metrics, statistics and figures

In [ ]:
!python summarize.py sim real lozo sens 2>&1 | tail -2
!mkdir -p figures && python fig_arch.py && python fig_case.py case mech && python fig_results.py forest alerts leads robust anchor coverage
!python fig_extra.py real eff 2>&1 | tail -1
import json, pandas as pd
S = json.load(open('results/summary.json'))
rows = {k: {m: v[m][0] for m in ['cindex', 'ibs', 'f1', 'ece', 'alert_miss', 'alert_lead_visible', 'lpb_coverage_events']}
        for k, v in S['sim_temporal'].items()}
pd.DataFrame(rows).T.round(4)

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('figures/*.png')):
    print(f); display(Image(f, width=700))